# 🧠 LLM Context Management and Dynamic System Prompts

---
**Notebook developed by** SzuLun Huang <szuh@berkeley.edu>  
**Under the guidance of** Eric Van Dusen <ericvd@berkeley.edu>  
**UC Berkeley, Data Science**

---

## 📖 The Story

**Prof. Eric** teaches Data 8 — UC Berkeley's intro data science course. This semester he has 300 students, each at a different skill level. He pulls Zoe aside after class:

> *"Zoe, I want to build an AI study assistant for my students. But here's the problem — a Data 8 beginner and a Data 100 student need completely different explanations for the same question. Can you make it adapt automatically?"*

Zoe says yes. She has no idea what she's gotten herself into.

**This notebook is her journey** — every Part is a new problem she runs into, and a new concept she learns to solve it. By the end, she has a production-ready design and Prof. Eric has his assistant.

---

## 🚀 How to Start

1. Click **Kernel** in the top menu
2. Select **Restart Kernel and Run All Cells**
3. Wait about **1–2 minutes** for the model to load ⏳
4. Then explore the interactive sections below! ✅

> ⚠️ You only need to do this **once** each time you open the notebook.


# 📦 Step 0: Setup — Imports & Model Loading (run once, ~1-2 min)

All dependencies and the local LLM are loaded here. Everything else depends on this cell.

In [1]:
# ═══════════════════════════════════════════════════════════
#  ALL IMPORTS — keep everything here for easy management
# ═══════════════════════════════════════════════════════════
import os
import json
import warnings
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

warnings.filterwarnings("ignore")

# ── Model path ──────────────────────────────────────────────
model_filename = "Llama-3.2-1B-Instruct-Q4_K_M.gguf"
model_path     = f"/home/jovyan/shared/{model_filename}"

print("🔍 Checking for model...")
print(f"   Path: {model_path}")
print()

if os.path.exists(model_path):
    print(f"✅ Model found!  Size: {os.path.getsize(model_path)/(1024**3):.2f} GB")
else:
    print("❌ Model not found! Please ask your teacher to check the shared folder.")
    raise FileNotFoundError(f"Model not found at {model_path}")

# ── Load model ──────────────────────────────────────────────
from llama_cpp import Llama

print("\n🔧 Loading model into memory (verbose=True so you can see tokens/sec)...")
print("⏳ This may take 1-2 minutes...\n")

n_ctx     = 4096
n_threads = 4

model = Llama(
    model_path=model_path,
    n_ctx=n_ctx,
    n_threads=n_threads,
    verbose=False  # ← shows tokens/sec and timing info during generation
)
clear_output(wait=True) 
print("="*60)
print("✅ Model loaded and ready!")
print("="*60)
print(f"   Model  : {model_filename}")
print(f"   Context: {n_ctx} tokens")
print(f"   Threads: {n_threads}")
print("\n   💡 Note: verbose=True lets you see how fast the model generates.")
print("   Watch for 'tokens/s' in the output — more tokens in context = slower!")


✅ Model loaded and ready!
   Model  : Llama-3.2-1B-Instruct-Q4_K_M.gguf
   Context: 4096 tokens
   Threads: 4

   💡 Note: verbose=True lets you see how fast the model generates.
   Watch for 'tokens/s' in the output — more tokens in context = slower!


# 🧠 Part 1: "Why Doesn't It Remember Anything?"

Zoe's first prototype is just a few lines: she types a question, the AI answers. It works. She's excited.

She sends Prof. Eric a demo. He replies: *"Great! Can I ask a follow-up question?"*

She tries it — and the AI has no idea what was just discussed.

After some digging through the docs, she finds the answer: the model is completely **stateless**. Every API call starts fresh. It only knows what you pass in *right now*:

```python
response = model(messages)  # the model only sees what you pass in right now
```

If Zoe wants her assistant to remember the conversation, **she** has to keep track of it and pass it back every time. The `messages` list is the model's entire world.

## What goes in a messages list?

Each message has a `role` and `content`:

| Role | Who it's from | When to use |
|------|--------------|-------------|
| `system` | Zoe (as the developer) | Set behaviour, tone, or rules |
| `user` | The student using the assistant | What they typed |
| `assistant` | The AI | Previous AI responses |

> 💡 Right now Zoe is testing the assistant herself, so she's playing both roles. Later, Prof. Eric's students will be the `user`.

**👇 Work through the 3 challenges — you'll feel exactly what Zoe discovered.**


In [3]:
import ipywidgets as widgets
import json as _json
from IPython.display import display, HTML, clear_output
import time

# ═══════════════════════════════════════════════════════════════════
#  Part 1 — Interactive Memory Explorer
# ═══════════════════════════════════════════════════════════════════

display(HTML("""
<style>
@keyframes fadeOut {
  from { opacity:1; transform:translateY(0);   max-height:120px; margin:5px 0; }
  to   { opacity:0; transform:translateY(-6px); max-height:0;    margin:0;     }
}
@keyframes fadeIn {
  from { opacity:0; transform:translateY(8px); }
  to   { opacity:1; transform:translateY(0);   }
}
@keyframes shake {
  0%,100%{transform:translateX(0)}
  20%{transform:translateX(-8px)} 40%{transform:translateX(8px)}
  60%{transform:translateX(-5px)} 80%{transform:translateX(5px)}
}
.bubble-wrap  { animation: fadeIn 0.35s ease both; }
.bubble-dying { animation: fadeOut 0.5s ease forwards; overflow:hidden; }
.amnesiac     { animation: shake 0.5s ease; }
.token-pill {
  display:inline-block; background:#313244; border-radius:20px;
  padding:2px 10px; font-size:0.76em; color:#a6adc8; margin-left:8px;
}
.token-pill.warn { background:#f9e2af22; color:#f9e2af; }
.token-pill.crit { background:#f38ba822; color:#f38ba8; }
</style>
"""))

# ── bubble renderer ──────────────────────────────────────────────────
def bubble(role, content, extra_class=""):
    cfg = {
        "system":    ("#f9e2af", "#f9e2af18", "left",  "⚙️ system"),
        "user":      ("#89b4fa", "#89b4fa18", "right", "👤 user"),
        "assistant": ("#a6e3a1", "#a6e3a118", "left",  "🤖 assistant"),
    }
    color, bg, side, label = cfg.get(role, ("#cdd6f4","#cdd6f418","left",role))
    align  = "flex-end"  if side == "right" else "flex-start"
    radius = "18px 18px 4px 18px" if side == "right" else "18px 18px 18px 4px"
    return f"""
    <div class="bubble-wrap {extra_class}"
         style="display:flex;justify-content:{align};margin:5px 0">
      <div style="max-width:75%">
        <div style="font-size:0.72em;color:{color};margin-bottom:3px;
                    text-align:{'right' if side=='right' else 'left'}">{label}</div>
        <div style="background:{bg};border:1px solid {color}44;
                    border-radius:{radius};padding:9px 14px;
                    color:#cdd6f4;font-size:0.87em;line-height:1.6">{content}</div>
      </div>
    </div>"""

def token_pill(msgs):
    try:
        n = sum(len(model.tokenize(m["content"].encode())) for m in msgs)
    except Exception:
        n = sum(len(m.get("content",""))//4 for m in msgs)
    cls = "crit" if n>500 else "warn" if n>150 else ""
    return f'<span class="token-pill {cls}">🪙 {n} tokens</span>'

# ── shared state ─────────────────────────────────────────────────────
state        = {"index": 0}
student_name = {"v": ""}
_c1_history  = {"msgs": []}   # stores the real C1 turn2 messages for C2 amnesia

# ── widgets ──────────────────────────────────────────────────────────
banner_out  = widgets.Output()
preview_out = widgets.Output()
output_area = widgets.Output()

c1_name = widgets.Text(placeholder="e.g. your name",
                       layout=widgets.Layout(width="280px"))
c1_like = widgets.Text(placeholder="e.g. Data 8 student, just starting with the datascience library",
                       layout=widgets.Layout(width="480px"))
c1_form = widgets.VBox([
    widgets.HTML('<div style="color:#a6adc8;font-size:0.84em;margin:10px 0 4px 0">'
                 '👤 Your name:</div>'),
    c1_name,
    widgets.HTML('<div style="color:#a6adc8;font-size:0.84em;margin:8px 0 4px 0">'
                 '📚 Something about yourself (course, background, what you\'re learning):</div>'),
    c1_like,
], layout=widgets.Layout(display="none", padding="0 0 10px 0"))

send_btn = widgets.Button(description="▶ Send", button_style="primary",
                          layout=widgets.Layout(width="110px", margin="10px 6px 0 0"))
next_btn = widgets.Button(description="Next →", button_style="info",
                          layout=widgets.Layout(width="110px", margin="10px 0 0 0"))

CHALLENGES = [
    {
        "id": "c1", "label": "Challenge 1 of 2",
        "title": "The AI remembers — because YOU gave it the memory",
        "color": "#89b4fa", "icon": "🧠",
        "instruction": "Enter your name and something about yourself, then hit <strong>Send</strong>.",
        "hint": "Every message in the list is part of the AI's memory. It can see all of them.",
        "form": "c1",
    },
    {
        "id": "c2", "label": "Challenge 2 of 2",
        "title": "Watch your memory disappear.",
        "color": "#f38ba8", "icon": "💀",
        "instruction": "Watch the messages vanish — then hit <strong>Send</strong> to confirm the AI forgot everything.",
        "hint": "The AI only sees what's in the list right now. No list = no memory.",
        "form": None,
    },
]

# ── banner & layout ───────────────────────────────────────────────────
def load_challenge(idx):
    c = CHALLENGES[idx]
    with banner_out:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#1e1e2e;border:2px solid {c['color']};
                    border-radius:12px;padding:16px 20px;margin:10px 0">
          <div style="display:flex;align-items:center;gap:10px;margin-bottom:8px">
            <span style="font-size:1.6em">{c['icon']}</span>
            <div>
              <div style="color:{c['color']};font-weight:bold;font-size:0.85em">{c['label']}</div>
              <div style="color:#cdd6f4;font-weight:bold;font-size:1em">{c['title']}</div>
            </div>
          </div>
          <div style="color:#cdd6f4;font-size:0.9em;margin-bottom:6px">{c['instruction']}</div>
          <div style="color:#a6adc8;font-size:0.8em">💡 {c['hint']}</div>
        </div>"""))

    c1_form.layout.display = "" if c["form"] == "c1" else "none"

    if c["id"] == "c1":
        refresh_c1_preview()
    else:
        # Show the real C1 history if available, else a placeholder
        saved = _c1_history.get("msgs")
        render_preview(saved if saved else [], label="📨 This was your memory from Challenge 1…")

    next_btn.disabled    = (idx == len(CHALLENGES) - 1)
    next_btn.description = "✅ Done" if idx == len(CHALLENGES) - 1 else "Next →"
    with output_area:
        clear_output()

# ── preview ───────────────────────────────────────────────────────────
def render_preview(messages, label=None):
    label = label or f'📨 Messages the model will receive {token_pill(messages)}'
    bubs  = "".join(bubble(m["role"], m["content"]) for m in messages)
    with preview_out:
        clear_output(wait=True)
        display(HTML(
            f'<div style="background:#1e1e2e;border-radius:10px;padding:14px">'
            f'<div style="color:#585b70;font-size:0.78em;margin-bottom:6px">{label}</div>'
            f'{bubs}</div>'))

def refresh_c1_preview(*_):
    name = c1_name.value.strip() or "you"
    like = c1_like.value.strip() or "…"
    # Preview shows what turn2 will look like (assistant bubble is a placeholder)
    msgs = [
        {"role": "system",    "content": "You are a friendly assistant. Keep replies to 1-2 sentences."},
        {"role": "user",      "content": f"Hi! My name is {name}. {like}."},
        {"role": "assistant", "content": "( AI will reply here after Turn 1 )"},
        {"role": "user",      "content": "What's my name and what can you tell me about my background?"},
    ]
    render_preview(msgs)

c1_name.observe(refresh_c1_preview, names="value")
c1_like.observe(refresh_c1_preview, names="value")

# ── amnesia animation ─────────────────────────────────────────────────
def play_amnesia_then_send():
    full_msgs = _c1_history.get("msgs", [])
    if not full_msgs:
        with output_area:
            clear_output()
            display(HTML('<p style="color:#f38ba8">⚠️ Please complete Challenge 1 first!</p>'))
        return

    # Erase bubbles one by one (skip last)
    for i in range(len(full_msgs) - 1):
        surviving = [
            bubble(m["role"], m["content"],
                   extra_class="bubble-dying" if j == i else "")
            for j, m in enumerate(full_msgs)
        ]
        with preview_out:
            clear_output(wait=True)
            display(HTML(
                '<div style="background:#1e1e2e;border-radius:10px;padding:14px">'
                '<div style="color:#f38ba8;font-size:0.78em;margin-bottom:6px">'
                '🗑️ Erasing memory…</div>'
                + "".join(surviving) + "</div>"))
        time.sleep(0.6)

    # Show only the lone question
    lone = [{"role": "user", "content": "What's my name and what can you tell me about my background?"}]
    with preview_out:
        clear_output(wait=True)
        display(HTML(
            '<div style="background:#1e1e2e;border-radius:10px;padding:14px">'
            f'<div style="color:#f38ba8;font-size:0.78em;margin-bottom:6px">'
            f'📨 All that remains {token_pill(lone)}</div>'
            + bubble(lone[0]["role"], lone[0]["content"]) + "</div>"))

    resp  = model.create_chat_completion(messages=lone, max_tokens=60, temperature=0.7)
    reply = resp["choices"][0]["message"]["content"].strip()
    name  = student_name["v"] or "you"

    with output_area:
        clear_output(wait=True)
        display(HTML(f"""
        <div class="amnesiac" style="background:#1e1e2e;border-radius:10px;
                    padding:14px;margin-top:8px">
          <div style="color:#585b70;font-size:0.78em;margin-bottom:8px">🤖 Model reply</div>
          {bubble("assistant", reply)}
          <div style="background:#f38ba822;border:2px solid #f38ba8;border-radius:10px;
               padding:12px 16px;margin-top:12px;text-align:center">
            <div style="font-size:1.4em;margin-bottom:4px">🫥</div>
            <div style="color:#f38ba8;font-weight:bold">Complete amnesia.</div>
            <div style="color:#a6adc8;font-size:0.82em;margin-top:6px;line-height:1.7">
              The model received <strong style="color:#f38ba8">1 message</strong> — no history, no name, nothing.<br>
              It has no idea who <strong style="color:#cdd6f4">{name}</strong> is.<br><br>
              <strong style="color:#cdd6f4">This is what Zoe's first broken prototype felt like.</strong><br>
              <span style="color:#585b70">The fix? Pass the full messages list every time. That's what Part 2 builds.</span>
            </div>
          </div>
        </div>"""))

# ── send logic ────────────────────────────────────────────────────────
def on_send(btn):
    c = CHALLENGES[state["index"]]
    send_btn.disabled    = True
    send_btn.description = "⏳ …"

    if c["id"] == "c2":
        with output_area:
            clear_output()
        play_amnesia_then_send()

    else:  # C1 — real two-turn conversation
        name = c1_name.value.strip()
        like = c1_like.value.strip()
        if not name or not like:
            with output_area:
                clear_output()
                display(HTML('<p style="color:#f38ba8">⚠️ Enter your name and something about yourself first!</p>'))
            send_btn.disabled    = False
            send_btn.description = "▶ Send"
            return
        student_name["v"] = name

        # Turn 1
        turn1 = [
            {"role": "system", "content": "You are a friendly assistant. Keep replies to 1-2 sentences."},
            {"role": "user",   "content": f"Hi! My name is {name}. {like}."},
        ]
        with output_area:
            clear_output()
            display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Turn 1: sending introduction…</p>'))
        resp1    = model.create_chat_completion(messages=turn1, max_tokens=60, temperature=0.7)
        ai_reply = resp1["choices"][0]["message"]["content"].strip()

        # Turn 2 with real AI reply in history
        turn2 = turn1 + [
            {"role": "assistant", "content": ai_reply},
            {"role": "user",      "content": "Based on what I just told you, what's my name and what can you tell me about my background?"},
        ]
        _c1_history["msgs"] = turn2   # save for C2 amnesia

        with output_area:
            clear_output()
            display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Turn 2: asking follow-up…</p>'))
        resp2 = model.create_chat_completion(messages=turn2, max_tokens=80, temperature=0.7)
        final = resp2["choices"][0]["message"]["content"].strip()

        bubs = "".join(bubble(m["role"], m["content"]) for m in turn2)
        with output_area:
            clear_output(wait=True)
            display(HTML(f"""
            <div style="background:#1e1e2e;border-radius:10px;padding:14px;margin-top:8px">
              <div style="color:#585b70;font-size:0.78em;margin-bottom:8px">
                📨 What the model received {token_pill(turn2)}</div>
              {bubs}
              <div style="border-top:1px solid #313244;margin:10px 0"></div>
              <div style="color:#585b70;font-size:0.78em;margin-bottom:6px">🤖 Model reply</div>
              {bubble("assistant", final)}
              <div style="background:#89b4fa22;border:1px solid #89b4fa44;border-radius:8px;
                   padding:10px 14px;margin-top:10px;font-size:0.82em;color:#a6adc8">
                ✅ The AI knows your name because <strong style="color:#cdd6f4">you gave it the memory</strong>.
                Hit <strong style="color:#89b4fa">Next →</strong> to see what happens when that memory disappears.
              </div>
            </div>"""))

    send_btn.disabled    = False
    send_btn.description = "▶ Send"

def on_next(btn):
    nxt = state["index"] + 1
    if nxt < len(CHALLENGES):
        state["index"] = nxt
        load_challenge(nxt)

send_btn.on_click(on_send)
next_btn.on_click(on_next)

# ── initial render ────────────────────────────────────────────────────
display(widgets.HTML("""
<div style="background:#1e1e2e;padding:16px 20px;border-radius:12px;margin-bottom:4px">
  <h3 style="color:#cdd6f4;margin:0 0 6px 0">🧠 Part 1: The messages List is the AI's Entire Memory</h3>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Two challenges. Each one lets you <em>feel</em> how AI memory works — and breaks.
  </p>
</div>
"""))
load_challenge(0)
display(banner_out, c1_form, preview_out,
        widgets.HBox([send_btn, next_btn]), output_area)


HTML(value='\n<div style="background:#1e1e2e;padding:16px 20px;border-radius:12px;margin-bottom:4px">\n  <h3 s…

Output()

Output()

Output()


# 👤 Part 2: "300 Students, 300 Different Needs"

Zoe shows the working prototype to Prof. Eric. He's happy — but immediately has a new request:

> *"This is great. But my Data 8 students are complete beginners — they need simple language and analogies. My Data 100 students are much more advanced — they'd find that patronising. Can the same assistant handle both?"*

Zoe's first instinct: write two different system prompts. But Prof. Eric has 300 students. She can't hardcode a prompt for each one.

Her solution: **store the student's info as a dict, and let code write the prompt automatically.**

```python
profile = {"name": "Student", "expertise": "beginner", ...}
system_prompt = build_system_message(profile)  # prompt writes itself
```

Change one field in the profile → the whole prompt regenerates. One function, any student.

**👇 Try the widget below** — edit the student profile on the left and watch the system prompt update instantly on the right.


In [4]:
import ipywidgets as widgets
import json as _json
from IPython.display import display, HTML, clear_output

# ── Profile → System Prompt builder ───────────────────────────
def build_system_message(profile, project_profile=None):
    name     = profile.get("name", "the user")
    expertise = profile.get("expertise", "intermediate")
    project  = profile.get("current_project", "")
    style    = profile.get("style_preferences", [])

    lines = [
        "You are a helpful AI assistant.",
        "",
        "## About the User",
        f"- Name: {name}",
        f"- Skill level: {expertise}",
    ]

    if project:
        lines.append(f"- Current project: {project}")

    if style:
        lines += ["", "## Response Style"]
        for s in style:
            lines.append(f"- {s}")

    if expertise.lower() in ["beginner", "new to coding"]:
        lines += [
            "",
            "## Important",
            "- Use simple language, avoid jargon",
            "- Always include a short code example",
            "- Explain each step clearly",
        ]
    elif expertise.lower() in ["expert", "senior", "advanced"]:
        lines += [
            "",
            "## Important",
            "- Be concise and technical",
            "- Skip basic explanations",
            "- Focus on edge cases and best practices",
        ]


    if project_profile:
        proj_name  = project_profile.get("name", "")
        proj_desc  = project_profile.get("description", "")
        proj_goal  = project_profile.get("current_goal", "")
        proj_tools = project_profile.get("tools", [])
        lines += ["", "## Current Project"]
        if proj_name:
            lines.append(f"- Project: {proj_name}" + (f" — {proj_desc}" if proj_desc else ""))
        if proj_goal:
            lines.append(f"- Goal: {proj_goal}")
        if proj_tools:
            lines.append(f"- Tools: {', '.join(proj_tools)}")

    return "\n".join(lines)

# ── Default profile ────────────────────────────────────────────
DEFAULT_PROFILE = {
    "name": "Alex",
    "expertise": "beginner",
    "current_project": "Climate Data Analysis with pandas",
    "style_preferences": [
        "Keep answers short and concise",
        "Always include code examples"
    ]
}

# ── Widgets ───────────────────────────────────────────────────
title = widgets.HTML("""
<div style="background:#1e1e2e;padding:16px 20px;border-radius:10px;margin-bottom:12px">
  <h3 style="color:#cdd6f4;margin:0;font-size:1.1em">👤 Part 2: User Profile → System Prompt</h3>
  <p style="color:#a6adc8;margin:6px 0 0 0;font-size:0.9em">
    Edit the profile on the left. Watch the system prompt update instantly.
    Then ask a question to see how the AI responds.
  </p>
</div>
""")

hint = widgets.HTML("""
<div style="background:#313244;padding:12px 16px;border-radius:8px;margin-bottom:14px;font-size:0.85em;color:#a6adc8">
  <div style="color:#f9e2af;font-weight:bold;margin-bottom:8px">🎯 Try this challenge:</div>
  <div style="line-height:2.0">
    <span style="color:#cdd6f4">Step 1:</span>
    Hit <strong style="color:#cdd6f4">Send</strong> with the default profile — read the response<br>
    <span style="color:#cdd6f4">Step 2:</span>
    Change <code style="background:#1e1e2e;padding:2px 6px;border-radius:3px;color:#f9e2af">expertise</code>
    from <code style="background:#1e1e2e;padding:2px 6px;border-radius:3px;color:#cdd6f4">"beginner"</code>
    to <code style="background:#1e1e2e;padding:2px 6px;border-radius:3px;color:#cdd6f4">"expert"</code>
    — watch the system prompt change<br>
    <span style="color:#cdd6f4">Step 3:</span>
    Hit <strong style="color:#cdd6f4">Send</strong> again — compare the two responses
  </div>
  <div style="margin-top:8px;color:#585b70;font-size:0.82em">
    Same question, same model — only the system prompt changed.
  </div>
</div>
""")

profile_label = widgets.HTML(
    '<p style="color:#89b4fa;font-weight:bold;margin:4px 0">① Edit the user profile:</p>'
)
profile_editor = widgets.Textarea(
    value=_json.dumps(DEFAULT_PROFILE, indent=2),
    layout=widgets.Layout(width="340px", height="220px")
)

arrow = widgets.HTML("""
<div style="display:flex;align-items:center;justify-content:center;padding:0 12px;color:#585b70;font-size:1.6em;margin-top:30px">→</div>
""")

prompt_label = widgets.HTML(
    '<p style="color:#a6e3a1;font-weight:bold;margin:4px 0">② Generated system prompt:</p>'
)
prompt_display = widgets.Textarea(
    value=build_system_message(DEFAULT_PROFILE),
    disabled=True,
    layout=widgets.Layout(width="340px", height="220px")
)

q_label = widgets.HTML(
    '<p style="color:#cdd6f4;font-weight:bold;margin:12px 0 4px 0">③ Question (pre-filled — or type your own):</p>'
)
q_input = widgets.Text(
    value="How do I read a CSV file in Python?",
    layout=widgets.Layout(width="520px")
)

send_btn = widgets.Button(
    description="▶ Send",
    button_style="primary",
    layout=widgets.Layout(width="90px", margin="8px 4px 0 0")
)
reset_btn = widgets.Button(
    description="🔄 Reset",
    button_style="warning",
    layout=widgets.Layout(width="90px", margin="8px 0 0 0")
)

output_area = widgets.Output()

# ── Live system prompt update ──────────────────────────────────
def on_profile_change(change):
    try:
        profile = _json.loads(profile_editor.value)
        prompt_display.value = build_system_message(profile)
        prompt_display.layout.border = "1px solid #a6e3a1"
    except _json.JSONDecodeError:
        prompt_display.value = "⚠️ Invalid JSON — fix the profile to see the system prompt"
        prompt_display.layout.border = "1px solid #f38ba8"

profile_editor.observe(on_profile_change, names="value")

# ── Render result ──────────────────────────────────────────────
def render_result(system_prompt, question, reply):
    return f"""
    <div style="background:#1e1e2e;border:2px solid #cba6f7;border-radius:10px;padding:14px;margin-top:12px">
      <div style="color:#cba6f7;font-weight:bold;margin-bottom:12px">🤖 AI Response</div>
      <div style="display:flex;gap:12px;margin-bottom:12px">
        <div style="flex:1">
          <div style="color:#a6adc8;font-size:0.8em;margin-bottom:4px">System prompt sent:</div>
          <div style="background:#313244;padding:8px;border-radius:5px;font-size:0.78em;
                      color:#cdd6f4;line-height:1.5;white-space:pre-wrap;
                      max-height:120px;overflow-y:auto">{system_prompt}</div>
        </div>
        <div style="flex:1">
          <div style="color:#a6adc8;font-size:0.8em;margin-bottom:4px">Question:</div>
          <div style="background:#313244;padding:8px;border-radius:5px;
                      font-size:0.88em;color:#cdd6f4">{question}</div>
        </div>
      </div>
      <div style="color:#a6adc8;font-size:0.8em;margin-bottom:4px">Response:</div>
      <div style="background:#313244;padding:10px;border-radius:6px;
                  color:#cdd6f4;font-size:0.9em;line-height:1.6">{reply}</div>
    </div>
    """

# ── Logic ──────────────────────────────────────────────────────
def on_send(btn):
    try:
        profile = _json.loads(profile_editor.value)
    except _json.JSONDecodeError:
        with output_area:
            clear_output()
            display(HTML('<p style="color:#f38ba8">⚠️ Fix the profile JSON first.</p>'))
        return

    question = q_input.value.strip()
    if not question:
        with output_area:
            clear_output()
            display(HTML('<p style="color:#f38ba8">⚠️ Please enter a question.</p>'))
        return

    system_prompt = build_system_message(profile)
    send_btn.disabled = True
    send_btn.description = "Running..."

    with output_area:
        clear_output()
        display(HTML('<p style="color:#89b4fa">⏳ Sending...</p>'))

    resp = model.create_chat_completion(
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": question},
        ],
        max_tokens=150,
        temperature=0.7
    )
    reply = resp["choices"][0]["message"]["content"].strip()

    send_btn.disabled = False
    send_btn.description = "▶ Send"

    with output_area:
        clear_output()
        display(HTML(render_result(system_prompt, question, reply)))

def on_reset(btn):
    profile_editor.value = _json.dumps(DEFAULT_PROFILE, indent=2)
    q_input.value = "How do I read a CSV file in Python?"
    with output_area:
        clear_output()

send_btn.on_click(on_send)
reset_btn.on_click(on_reset)

display(
    title,
    hint,
    widgets.HBox([
        widgets.VBox([profile_label, profile_editor]),
        arrow,
        widgets.VBox([prompt_label, prompt_display]),
    ]),
    q_label,
    widgets.HBox([q_input, send_btn, reset_btn]),
    output_area
)

HTML(value='\n<div style="background:#1e1e2e;padding:16px 20px;border-radius:10px;margin-bottom:12px">\n  <h3 …

HTML(value='\n<div style="background:#313244;padding:12px 16px;border-radius:8px;margin-bottom:14px;font-size:…

HTML(value='<p style="color:#cdd6f4;font-weight:bold;margin:12px 0 4px 0">③ Question (pre-filled — or type you…

Output()

## 🎓 Demo: Data 8 vs Data 100 

Same question, two very different students:

| Course | Student type | What they need |
|--------|-------------|----------------|
| **Data 8** | Complete beginner | Simple language, analogies, `datascience` library |
| **Data 100** | Experienced | Concise, technical, `pandas` / industry tools |

**What to watch:** Ask the AI *"How do I read a CSV file?"* — but change the `course` field.

> 🔍 The model is identical. The question is identical. Only the system prompt changes — but the answer is completely different. This is exactly what Zoe builds for Prof. Eric.


In [10]:
# ── Data 8 vs Data 100 System Prompt Builder ─────────────────────────
# Requires: `model` already loaded in a previous cell (Llama object from llama_cpp)
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

def build_course_prompt(course):
    """Return a tailored system prompt for Data 8 or Data 100 students."""
    if course == "Data 8":
        lib_guidance = (
            "The student is in Data 8 (Foundations of Data Science) at UC Berkeley. "
            "This course uses ONLY Python. "
            "ALWAYS use the `datascience` package. The correct syntax is: "
            "from datascience import Table; t = Table.read_table('file.csv'). "
            "Never mention R, pandas, or any other language or library."
        )
    else:
        lib_guidance = (
            "The student is in Data 100 (Principles and Techniques of Data Science). "
            "Always recommend `pandas` (pd.DataFrame, pd.read_csv, etc.). "
            "You may use technical terminology; the student knows Python."
        )
    return (
        "You are a helpful TA for UC Berkeley's data science program.\n"
        f"{lib_guidance}\n"
        "Keep answers concise and include a short code snippet."
    )

question = "How do I read a CSV file?"

COURSE_COLORS = {
    "Data 8":   {"border": "#a6e3a1", "label": "#a6e3a1", "keyword": "datascience", "kw_color": "#a6e3a1"},
    "Data 100": {"border": "#89b4fa", "label": "#89b4fa", "keyword": "pandas",      "kw_color": "#89b4fa"},
}

def highlight(text, keyword, color):
    """Wrap keyword occurrences in a coloured <span>."""
    return text.replace(
        keyword,
        f'<span style="color:{color};font-weight:bold;background:#1e1e2e;'
        f'padding:1px 5px;border-radius:3px">{keyword}</span>'
    )

output_area = widgets.Output()

def run_comparison(_=None):
    run_btn.disabled = True
    run_btn.description = "⏳ Running..."

    with output_area:
        clear_output()
        display(HTML("""
        <div style="color:#a6adc8;font-size:0.88em;padding:8px 0">
          ⏳ Querying model for both courses... this may take a few seconds.
        </div>
        """))

    results = {}
    for course, cfg in COURSE_COLORS.items():
        sys_prompt = build_course_prompt(course)
        resp = model.create_chat_completion(
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user",   "content": question},
            ],
            max_tokens=150,
            temperature=0.7,
        )
        results[course] = resp["choices"][0]["message"]["content"].strip()

    with output_area:
        clear_output()
        html_parts = ['<div style="display:flex;gap:14px;margin-top:10px">']
        for course, cfg in COURSE_COLORS.items():
            reply_html = highlight(results[course], cfg["keyword"], cfg["kw_color"])
            # also highlight the Table.read_table / pd.read_csv call for extra clarity
            reply_html = reply_html.replace(
                "Table.read_table",
                f'<span style="color:{cfg["kw_color"]};font-weight:bold">Table.read_table</span>'
            ).replace(
                "pd.read_csv",
                f'<span style="color:{cfg["kw_color"]};font-weight:bold">pd.read_csv</span>'
            )
            html_parts.append(f"""
            <div style="flex:1;background:#1e1e2e;border:2px solid {cfg['border']};
                        border-radius:10px;padding:14px">
              <div style="color:{cfg['label']};font-weight:bold;font-size:1em;margin-bottom:8px">
                🎓 {course}
              </div>
              <div style="color:#a6adc8;font-size:0.78em;margin-bottom:6px">
                System prompt tells the AI to use:
                <span style="color:{cfg['kw_color']};font-weight:bold">{cfg['keyword']}</span>
              </div>
              <div style="background:#313244;padding:10px;border-radius:6px;
                          color:#cdd6f4;font-size:0.87em;line-height:1.6;white-space:pre-wrap">
                {reply_html}
              </div>
            </div>""")
        html_parts.append('</div>')
        html_parts.append("""
        <div style="margin-top:12px;background:#313244;padding:10px 14px;border-radius:8px;
                    font-size:0.85em;color:#a6adc8">
          ✨ <strong style="color:#f9e2af">Key observation:</strong>
          Same model, same question — only the system prompt changed.<br>
          The highlighted library name shows exactly where the behaviour diverged.
        </div>""")
        display(HTML("".join(html_parts)))

    run_btn.disabled = False
    run_btn.description = "▶ Run Again"

run_btn = widgets.Button(
    description="▶ Run Comparison",
    button_style="primary",
    layout=widgets.Layout(width="160px", margin="0 0 10px 0")
)
run_btn.on_click(run_comparison)

display(widgets.HTML("""
<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">
  <h4 style="color:#cdd6f4;margin:0 0 6px 0">🎓 Data 8 vs Data 100 — Side-by-Side Comparison</h4>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Question: <em>"How do I read a CSV file?"</em><br>
    Watch how the <strong style="color:#f9e2af">highlighted library</strong> changes
    based on the system prompt alone.
  </p>
</div>
"""), run_btn, output_area)

HTML(value='\n<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">\n  <h4 …

Button(button_style='primary', description='▶ Run Comparison', layout=Layout(margin='0 0 10px 0', width='160px…

Output()



## 🪙 Token Counter: The Hidden Cost Zoe Didn't Expect

The assistant is working. Prof. Eric is happy. But a week into the semester, Zoe notices something: **the assistant is getting slower.**

Every message — system prompt, chat history, new question — consumes **tokens** from the model's context window (4,096 tokens for our local model). More tokens in = longer wait.

| What gets sent | Approx. tokens |
|---------------|----------------|
| Simple question only | ~8 tokens |
| Question + system prompt | ~100 tokens |
| Question + system prompt + 20–30 turns of history | ~1,000–3,000 tokens |

Run the cell below to see the **exact numbers** for each scenario.

> 💡 This is why **history compression** (Part 3) is not optional — especially on a local 1B model where every extra token adds real waiting time for students.


In [2]:
# ── Token Counter Demonstration ──────────────────────────────────────
# llama-cpp-python exposes tokenize() so we can count tokens exactly.

def count_tokens(messages):
    """Count total tokens across all messages in a list."""
    return sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in messages)

def estimated_wait(tokens, speed_tps=25):
    """Rough wait estimate: context tokens / generation speed."""
    return tokens / speed_tps

simple_question  = "How do I read a CSV file in Python?"
data100_prompt   = build_course_prompt("Data 100")

msgs_simple = [
    {"role": "user", "content": simple_question}
]
msgs_with_prompt = [
    {"role": "system", "content": data100_prompt},
    {"role": "user",   "content": simple_question},
]
msgs_with_history = [
    {"role": "system", "content": data100_prompt},
    {"role": "user",      "content": "Hi! I'm Zoe, a student from Taiwan studying AI at UC Berkeley."},
    {"role": "assistant", "content": "Welcome Zoe! That's exciting. What are you working on?"},
    {"role": "user",      "content": "I'm learning about context management in LLMs for a class project."},
    {"role": "assistant", "content": "Great topic! Context management is key for building real AI apps."},
    {"role": "user",      "content": "I'm also learning pandas and scikit-learn for the data side."},
    {"role": "assistant", "content": "Nice combo. Are you using Jupyter or a local environment?"},
    {"role": "user",   "content": simple_question},
]

t1 = count_tokens(msgs_simple)
t2 = count_tokens(msgs_with_prompt)
t3 = count_tokens(msgs_with_history)

SPEED = 25  # tokens/sec — typical for Llama-3.2-1B on CPU

rows = [
    ("Simple question only",              t1, msgs_simple),
    ("+ System prompt",                   t2, msgs_with_prompt),
    ("+ 6-turn history preview",          t3, msgs_with_history),
]

from IPython.display import display, HTML

html = ['<div style="background:#1e1e2e;border-radius:10px;padding:16px 20px;margin-top:8px">']
html.append('<h4 style="color:#cdd6f4;margin:0 0 12px 0">🪙 Token Count & Estimated Wait Time</h4>')
html.append('<table style="width:100%;border-collapse:collapse;font-size:0.88em">')
html.append('''<tr style="color:#a6adc8;border-bottom:1px solid #45475a">
  <th style="text-align:left;padding:6px 10px">Scenario</th>
  <th style="text-align:right;padding:6px 10px">Tokens</th>
  <th style="text-align:right;padding:6px 10px">% of 4096</th>
  <th style="text-align:right;padding:6px 10px">Est. wait (@25 tok/s)</th>
</tr>''')

bar_colors = ["#a6e3a1", "#89b4fa", "#f38ba8"]
for (label, t, _), color in zip(rows, bar_colors):
    pct  = t / 4096 * 100
    wait = estimated_wait(t, SPEED)
    bar_w = max(4, int(pct * 1.5))
    html.append(f'''<tr style="border-bottom:1px solid #313244">
  <td style="padding:8px 10px;color:#cdd6f4">{label}</td>
  <td style="text-align:right;padding:8px 10px;color:{color};font-weight:bold">{t}</td>
  <td style="text-align:right;padding:8px 10px">
    <span style="display:inline-block;width:{bar_w}px;height:10px;
                 background:{color};border-radius:3px;vertical-align:middle"></span>
    <span style="color:#a6adc8;margin-left:6px">{pct:.1f}%</span>
  </td>
  <td style="text-align:right;padding:8px 10px;color:#f9e2af">~{wait:.1f} s</td>
</tr>''')

html.append('</table>')
html.append(f'''<div style="margin-top:12px;padding:10px 14px;background:#313244;border-radius:8px;
                            color:#a6adc8;font-size:0.85em;line-height:1.8">
  💡 <strong style="color:#f9e2af">Cost us time:</strong>
  Adding a system prompt multiplies token count by ~{t2/t1:.0f}×.<br>
  Adding 6 turns of history multiplies it by ~{t3/t1:.0f}× vs. the bare question.<br>
  With 25 full turns (Zoe's story below), expect <strong style="color:#f38ba8">1,000 + tokens</strong>
  and ~{1000/SPEED:.0f}+ seconds of wait time on this local 1B model.<br>
  👉 <strong style="color:#cdd6f4">This is exactly why history compression (Part 3) matters.</strong>
</div>
</div>''')

display(HTML("".join(html)))


NameError: name 'build_course_prompt' is not defined


## 📖 Part 2b: How the Assistant Learns Who Zoe Is

Prof. Eric asks: *"Can the assistant learn a student's background just from conversation — without making them fill out a form?"*

Yes. Instead of asking *"What's your skill level?"* upfront, the assistant can **infer the profile from what the student says** across multiple turns.

Below is a **simulated 25-turn conversation** with Zoe herself as the student. Her background — Taiwan, AI focus, pandas experience — emerges naturally through the chat.

**What to observe:**
- Early turns: Zoe is just asking questions, profile is mostly empty
- Middle turns: the AI starts tailoring responses to her background
- Later turns: the AI knows her skills, goals, and style without ever being told explicitly

Run the cell and watch how the system prompt grows with the history.


In [15]:
# ── Zoe's Storyline History (25 turns) ──────────────────────────────
# This conversation was designed to reveal Zoe's profile naturally.
# It demonstrates how context shapes model behaviour over time.

# ── Fallbacks (in case earlier cells didn't run) ─────────────────────
if "simple_question" not in dir():
    simple_question = "What should I focus on to get better at AI?"

if "count_tokens" not in dir():
    def count_tokens(messages):
        """Rough token estimate: ~4 chars per token."""
        total_chars = sum(len(m.get("content", "")) for m in messages)
        return total_chars // 4

if "n_ctx" not in dir():
    n_ctx = 4096  # default — matches the model-loading cell

# ─────────────────────────────────────────────────────────────────────

ZOE_HISTORY = [
    {"role": "user",      "content": "Hi! I'm Zoe. I'm from Taiwan and I'm studying at UC Berkeley."},
    {"role": "assistant", "content": "Welcome Zoe! Great to meet you. What are you studying?"},
    {"role": "user",      "content": "I'm focusing on AI and machine learning. It's my first semester here."},
    {"role": "assistant", "content": "That's exciting! Berkeley has a great CS program. What draws you to AI?"},
    {"role": "user",      "content": "I want to understand how language models work — especially memory and context."},
    {"role": "assistant", "content": "Context management is a fascinating area. Are you working on a project?"},
    {"role": "user",      "content": "Yes, I'm building a teaching notebook about LLM context management for a class."},
    {"role": "assistant", "content": "That sounds like a great project. What tools are you using?"},
    {"role": "user",      "content": "Python, Jupyter, and llama-cpp-python with a local Llama model."},
    {"role": "assistant", "content": "Nice setup! Running locally avoids API costs. How's the performance?"},
    {"role": "user",      "content": "It's a bit slow — the 1B model takes a few seconds per response."},
    {"role": "assistant", "content": "That's expected. Reducing token count helps. Have you tried history compression?"},
    {"role": "user",      "content": "Not yet. That's actually one of the topics I want to teach in the notebook."},
    {"role": "assistant", "content": "Perfect timing then. Summarization and entity extraction are two common approaches."},
    {"role": "user",      "content": "I also want to show students the difference between Data 8 and Data 100 workflows."},
    {"role": "assistant", "content": "Good idea. The datascience vs pandas distinction is a classic Berkeley contrast."},
    {"role": "user",      "content": "Exactly! I think system prompts are the key to making that demo work."},
    {"role": "assistant", "content": "Right — change one field in the profile, the whole prompt regenerates."},
    {"role": "user",      "content": "I'm also learning about token budgets. Every token = more wait time locally."},
    {"role": "assistant", "content": "That's a great insight to teach. Students often ignore token costs until they feel it."},
    {"role": "user",      "content": "My background is more stats than CS, so I'm still getting comfortable with Python."},
    {"role": "assistant", "content": "Stats is a great foundation. pandas and numpy will feel natural to you."},
    {"role": "user",      "content": "I also want a fun mode where the AI speaks in a playful, encouraging tone."},
    {"role": "assistant", "content": "Easy — just add that as a style option in the system prompt."},
    {"role": "user",      "content": "Cool, I think that will make the notebook more fun. Thanks for helping me plan this!"},
]

# ── Show history with token count ────────────────────────────────────
msgs_full_zoe = [
    {"role": "system", "content": "You are a helpful AI assistant for Zoe, a student from Taiwan studying AI at UC Berkeley."},
    *ZOE_HISTORY,
    {"role": "user",   "content": simple_question},
]

t_full = count_tokens(msgs_full_zoe)

print(f"📖 Zoe's history: {len(ZOE_HISTORY)} messages across {len(ZOE_HISTORY)//2} turns")
print(f"🪙 Token count with full history: {t_full} tokens ({t_full/n_ctx*100:.1f}% of context window)")
print()
print("─"*60)
print("💡 What Zoe's profile reveals across the conversation:")
print("   Name        : Zoe")
print("   Origin      : Taiwan")
print("   School      : UC Berkeley (first semester)")
print("   Focus       : AI / LLM context management")
print("   Tools       : Python, Jupyter, llama-cpp-python")
print("   Background  : Data Science (not pure CS)")
print("   Goal        : Build a teaching notebook")
print("   Fun mode    : Different response styles (senior, coach, professor)")
print("─"*60)
print()
print("Now send a question WITH this history to see how the model responds differently:")

# ── Demo: ask a question with Zoe's full history ─────────────────────
resp = model.create_chat_completion(
    messages=msgs_full_zoe,
    max_tokens=150,
    temperature=0.7
)
print()
print("🤖 AI Response (with Zoe's full history as context):")
print(resp["choices"][0]["message"]["content"].strip())
print()
print("✨ Compare this to Scenario 1 (simple question only) from the token counter above.")
print("   The model now knows Zoe's background and tailors the response accordingly.")

📖 Zoe's history: 25 messages across 12 turns
🪙 Token count with full history: 457 tokens (11.2% of context window)

────────────────────────────────────────────────────────────
💡 What Zoe's profile reveals across the conversation:
   Name        : Zoe
   Origin      : Taiwan
   School      : UC Berkeley (first semester)
   Focus       : AI / LLM context management
   Tools       : Python, Jupyter, llama-cpp-python
   Background  : Data Science (not pure CS)
   Goal        : Build a teaching notebook
   Fun mode    : Different response styles (senior, coach, professor)
────────────────────────────────────────────────────────────

Now send a question WITH this history to see how the model responds differently:

🤖 AI Response (with Zoe's full history as context):
To read a CSV file in Python, you can use the `csv` module. Here's a simple example:

```python
import csv

with open('your_file.csv', 'r') as file:
    reader = csv.reader(file)
    for row in reader:
        print(row)
```

Rep

## 🎨 Bonus: Change the AI's Style with One Line

System prompts don't just control **content** — they control **tone and style** too.  
Here we swap a single instruction to show how the same answer sounds completely different depending on who's "speaking".

> 💡 **What to notice:** Same question, same model, same facts — only the style instruction changes.


In [7]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Style options ────────────────────────────────────────────
STYLES = {
    "👤 Friendly Senior":   "You are a helpful senior student. Answer in 1-2 sentence only, like you're explaining to a friend.",
    "🎉 Encouraging Coach": "Respond in an upbeat tone. Answer in 1-2 sentence only, then add one short encouragement.",
    "🎓 Professor":         "Respond formally. Answer in 1-2 sentence only, using precise terminology.",
}
   

TOPIC = "What is a context window in LLMs, and why should I care?"

# ── Widgets ───────────────────────────────────────────────────
style_dropdown = widgets.Dropdown(
    options=list(STYLES.keys()),
    description="Style:",
    layout=widgets.Layout(width="280px"),
)
send_btn = widgets.Button(
    description="▶ Try this style",
    button_style="primary",
    layout=widgets.Layout(width="160px", margin="0 0 0 10px"),
)
output_area = widgets.Output()

display(widgets.HTML(f"""
<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">
  <h4 style="color:#cdd6f4;margin:0 0 6px 0">🎨 Same Question, Different Style</h4>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Pick a style, hit <strong>Try this style</strong>. One call, instant result.<br>
    <em>Question: "{TOPIC}"</em>
  </p>
</div>
"""))

display(widgets.HBox([style_dropdown, send_btn]))
display(output_area)

def on_send(btn):
    send_btn.disabled = True
    send_btn.description = "⏳ …"
    style_name = style_dropdown.value
    instruction = STYLES[style_name]

    with output_area:
        clear_output()
        display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Asking model...</p>'))

    resp = model.create_chat_completion(
        messages=[
            {"role": "system", "content": instruction},
            {"role": "user",   "content": TOPIC},
        ],
        max_tokens=60,
        temperature=0.8,
    )
    reply = resp["choices"][0]["message"]["content"].strip()

    with output_area:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#1e1e2e;border:2px solid #cba6f7;border-radius:10px;padding:14px;margin-top:4px">
          <div style="color:#cba6f7;font-size:0.8em;margin-bottom:6px">
            Style: <strong style="color:#cdd6f4">{style_name}</strong>
          </div>
          <div style="background:#313244;padding:10px 14px;border-radius:8px;
                      color:#cdd6f4;font-size:0.9em;line-height:1.7">
            {reply}
          </div>
          <div style="margin-top:8px;color:#585b70;font-size:0.78em">
            💡 Now pick a different style and compare — same question, same model.
          </div>
        </div>
        """))

    send_btn.disabled = False
    send_btn.description = "▶ Try this style"

send_btn.on_click(on_send)

HTML(value='\n<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">\n  <h4 …

Output()

# 🗜️ Part 3: "The Assistant Is Getting Slower Every Day"

Two weeks into the semester, Prof. Eric messages Zoe:

> *"Students are complaining the assistant takes forever to respond. What's going on?"*

Zoe checks the logs. Some students have been chatting for 40+ turns. The context window is nearly full — and every call has to process the entire history from scratch.

She needs a way to **shrink old history without losing important information.**

## Part 3a: Three Approaches to History Compression

Zoe's first idea is simple: just delete old messages. But that causes amnesia — the model forgets everything the student said earlier.

Her solution: **compress** the old turns into a shorter summary, then keep that summary in context instead of the raw messages.

There are three ways to do this:

| Strategy | What it does | Trade-off |
|---|---|---|
| **Full Text Summarization** | Rewrites old turns as 1-2 sentences | Simple, but may lose specific details |
| **Key Entity Extraction** | Pulls out names, tools, decisions as bullet points | Token-efficient, but misses narrative flow |
| **Semantic Compression** | Combines both — one summary + key bullets | Best quality, costs one extra model call |

**👇 Pick a strategy below and see what it produces from the same 5-turn conversation.**


In [11]:
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# ── Three compression strategies ─────────────────────────────
def summarize_full_text(messages_to_summarize):
    """Summarize entire conversation into prose"""
    conversation_text = ""
    for msg in messages_to_summarize:
        role = "User" if msg["role"] == "user" else "AI"
        conversation_text += f"[{role}]: {msg['content']}\n\n"
    prompt = f"""Summarize this conversation in 1-2 sentences only:\n\n{conversation_text}"""
    response = model.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=60,
        temperature=0.7
    )
    return response['choices'][0]['message']['content']

def extract_key_entities(messages_to_summarize):
    """Extract important facts and decisions"""
    conversation_text = ""
    for msg in messages_to_summarize:
        role = "User" if msg["role"] == "user" else "AI"
        conversation_text += f"[{role}]: {msg['content']}\n\n"
    prompt = f"""Extract key facts from this conversation as 3-4 bullet points only:\n\n{conversation_text}\nFormat as: - Fact\n- Fact"""
    response = model.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0.3
    )
    return response['choices'][0]['message']['content']

def compress_semantic(messages_to_summarize):
    """Combine both approaches for optimal balance"""
    conversation_text = ""
    for msg in messages_to_summarize:
        role = "User" if msg["role"] == "user" else "AI"
        conversation_text += f"[{role}]: {msg['content']}\n\n"
    prompt = f"""Compress this conversation:\n\n{conversation_text}\nProvide:\n1. One-sentence summary\n2. Key facts (2-3 bullets only)"""
    response = model.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=80,
        temperature=0.5
    )
    return response['choices'][0]['message']['content']

# ── Sample conversation to compress ──────────────────────────
sample_conversation = [
    {"role": "user",      "content": "I'm building a climate data analysis project"},
    {"role": "assistant", "content": "That sounds interesting! What tools are you using?"},
    {"role": "user",      "content": "I'm using Python with pandas. But should I use polars?"},
    {"role": "assistant", "content": "Pandas is good for beginners. Polars is faster but harder."},
    {"role": "user",      "content": "I'll stick with pandas for now"},
]

STRATEGIES = {
    "📝 Full Text Summarization": {
        "fn":      summarize_full_text,
        "best":    "Narrative understanding",
        "problem": "May lose specific details",
        "color":   "#89b4fa",
    },
    "🏷️ Key Entity Extraction": {
        "fn":      extract_key_entities,
        "best":    "Preserving specific facts",
        "problem": "May miss context",
        "color":   "#a6e3a1",
    },
    "🔀 Semantic Compression (recommended)": {
        "fn":      compress_semantic,
        "best":    "Balanced coverage",
        "problem": "Costs one extra model call",
        "color":   "#cba6f7",
    },
}

# ── Widgets ───────────────────────────────────────────────────
strategy_dropdown = widgets.Dropdown(
    options=list(STRATEGIES.keys()),
    description="Strategy:",
    layout=widgets.Layout(width="340px"),
)
send_btn = widgets.Button(
    description="▶ Try this strategy",
    button_style="primary",
    layout=widgets.Layout(width="180px", margin="0 0 0 10px"),
)
output_area = widgets.Output()

display(widgets.HTML("""
<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">
  <h4 style="color:#cdd6f4;margin:0 0 6px 0">🗜️ Three Compression Strategies</h4>
  <p style="color:#a6adc8;margin:0;font-size:0.88em">
    Pick a strategy and hit <strong>Try this strategy</strong>.<br>
    Each one compresses the same 5-turn conversation differently.
  </p>
</div>
"""))
display(widgets.HBox([strategy_dropdown, send_btn]))
display(output_area)

def on_send(btn):
    send_btn.disabled = True
    send_btn.description = "⏳ …"
    name = strategy_dropdown.value
    cfg  = STRATEGIES[name]

    with output_area:
        clear_output()
        display(HTML('<p style="color:#89b4fa;font-size:0.88em">⏳ Compressing...</p>'))

    result = cfg["fn"](sample_conversation)

    with output_area:
        clear_output(wait=True)
        display(HTML(f"""
        <div style="background:#1e1e2e;border:2px solid {cfg['color']};border-radius:10px;padding:14px;margin-top:4px">
          <div style="color:{cfg['color']};font-weight:bold;margin-bottom:10px">{name}</div>
          <div style="background:#313244;padding:10px 14px;border-radius:8px;
                      color:#cdd6f4;font-size:0.9em;line-height:1.7;white-space:pre-wrap">{result}</div>
          <div style="display:flex;gap:20px;margin-top:10px;font-size:0.82em">
            <span>✅ <span style="color:#a6adc8">Best for: </span>
                  <span style="color:#cdd6f4">{cfg['best']}</span></span>
            <span>❌ <span style="color:#a6adc8">Problem: </span>
                  <span style="color:#f38ba8">{cfg['problem']}</span></span>
          </div>
        </div>
        """))

    send_btn.disabled = False
    send_btn.description = "▶ Try this strategy"

send_btn.on_click(on_send)

HTML(value='\n<div style="background:#1e1e2e;padding:14px 18px;border-radius:10px;margin-bottom:10px">\n  <h4 …

Output()

# ⚠️ Part 3b: What Zoe Almost Did Wrong

Before finding the right solution, Zoe tries a few things that seem reasonable — and discovers why each one fails.

These are the three most common mistakes developers make with LLM context. Each one has a measurable cost.

> 🎯 **Teaching goal:** Every anti-pattern below produces either a TokenError, degraded quality, or silent slowness. Run the cell to *feel* the difference.

<details>
<summary style="color:#f38ba8;cursor:pointer;font-size:0.95em">👉 Why does this matter for a local 1B model?</summary>

On a cloud API (GPT-4, Claude), bad context design costs **money**.
On our local Llama-3.2-1B, it costs **time** — students will sit waiting.
The feedback is immediate and visceral, which makes this the perfect environment to learn context discipline.

</details>


In [12]:
# ── Anti-Pattern Demonstration ───────────────────────────────────────
from IPython.display import display, HTML

SPEED = 25   # tokens/sec
N_CTX = 4096

def tok(text):
    return len(model.tokenize(text.encode("utf-8")))

def count_msgs(msgs):
    return sum(tok(m["content"]) for m in msgs)

# ── Anti-Pattern 1: Dump entire history without compression ───────────
raw_history = list(ZOE_HISTORY) * 2   # simulate a longer session
ap1_msgs = (
    [{"role": "system", "content": "You are a helpful assistant."}]
    + raw_history
    + [{"role": "user", "content": "What should I do next?"}]
)
ap1_tok = count_msgs(ap1_msgs)

# ── Anti-Pattern 2: Bloated system prompt with irrelevant rules ───────
bloated_lines = [
    "You are a helpful AI assistant for UC Berkeley students.",
    "Always be polite. Never be rude. Always use proper grammar. Do not use slang.",
    "Remember to be helpful. Be concise. But also be thorough. Include examples.",
    "Do not make things up. Always cite sources when possible. Be encouraging.",
    "Use bullet points when appropriate. Avoid passive voice where possible.",
    "Remember: the student is always right. Treat every question with respect.",
    "The student may be stressed. Be empathetic. Consider cultural differences.",
    "Do not assume gender. Use inclusive language. Be aware of accessibility.",
    "Always end with a summary. Start with the most important point.",
    "If you are unsure, say so. If you know, say so confidently.",
    "Current date: 2025. Location: Berkeley, CA. Language: English (default).",
]
bloated_system = "\n".join(bloated_lines) + "\nExtra padding: " + "x " * 200

ap2_msgs = [
    {"role": "system", "content": bloated_system},
    {"role": "user",   "content": "How do I read a CSV file?"},
]
ap2_tok = count_msgs(ap2_msgs)

# ── Anti-Pattern 3: Raw DB dump injected into context ─────────────────
fake_db_rows = [
    {
        "id": i,
        "user": "zoe@berkeley.edu",
        "timestamp": "2025-01-01T00:00:00Z",
        "session_id": "sess_" + str(i).zfill(4),
        "event": "page_view",
        "page": "/page/" + str(i),
        "metadata": {"browser": "Chrome", "os": "macOS", "ip": "10.0.0." + str(i % 255)},
    }
    for i in range(30)
]
fake_db_dump = str(fake_db_rows)

ap3_msgs = [
    {"role": "system", "content": "You are a helpful assistant. Here is the user activity log: " + fake_db_dump},
    {"role": "user",   "content": "What courses should I take next semester?"},
]
ap3_tok = count_msgs(ap3_msgs)

# ── Good practice: minimal, targeted context ──────────────────────────
good_system = (
    "You are a helpful TA for UC Berkeley's data science program. "
    "The student (Zoe, from Taiwan) is in Data 100, learning LLM context management. "
    "Keep answers concise and include code examples."
)
good_msgs = [
    {"role": "system", "content": good_system},
    {"role": "user",   "content": "How do I read a CSV file?"},
]
good_tok = count_msgs(good_msgs)

# ── HTML output ───────────────────────────────────────────────────────
PATTERNS = [
    {
        "label":   "Anti-Pattern 1",
        "icon":    "❌",
        "title":   "Dumping full history without compression",
        "code":    "messages = system + ZOE_HISTORY * 2 + [user_question]",
        "problem": "History grows unbounded. At 100 turns the context window overflows; model throws an error or silently drops early turns.",
        "tokens":  ap1_tok,
        "color":   "#f38ba8",
        "fix":     "Use sliding window or summary compression (Part 3).",
    },
    {
        "label":   "Anti-Pattern 2",
        "icon":    "❌",
        "title":   "Bloated system prompt with generic rules",
        "code":    "system = 'Be polite. Be concise. Be thorough...' x 50 lines",
        "problem": "Wastes tokens on instructions the model follows by default. Dilutes the specific, important instructions you actually care about.",
        "tokens":  ap2_tok,
        "color":   "#fab387",
        "fix":     "Keep system prompt under 150 tokens. Be specific, not generic.",
    },
    {
        "label":   "Anti-Pattern 3",
        "icon":    "❌",
        "title":   "Raw database / file dump into context",
        "code":    "system += str(db.query('SELECT * FROM logs LIMIT 30'))",
        "problem": "30 JSON rows is 800+ tokens of irrelevant fields (IP, browser, session_id). The useful signal is buried in noise.",
        "tokens":  ap3_tok,
        "color":   "#f9e2af",
        "fix":     "Extract only relevant fields. Use RAG to retrieve only what is needed (Part 3c below).",
    },
    {
        "label":   "Good Practice",
        "icon":    "✅",
        "title":   "Minimal, targeted system prompt",
        "code":    "system = 3-sentence profile + 1 behavioural rule",
        "problem": "—",
        "tokens":  good_tok,
        "color":   "#a6e3a1",
        "fix":     "This is the baseline. Extend only when you have a clear reason.",
    },
]

cards = ""
for p in PATTERNS:
    wait    = p["tokens"] / SPEED
    bar_w   = max(3, int(p["tokens"] / N_CTX * 300))
    pct_str = "{:.1f}".format(p["tokens"] / N_CTX * 100)
    wait_str = "{:.1f}".format(wait)
    fix_html = (
        "" if p["problem"] == "—"
        else '<div style="margin-top:8px;font-size:0.83em">'
             '<span style="color:#585b70">Fix: </span>'
             '<span style="color:#a6e3a1">' + p["fix"] + '</span></div>'
    )
    cards += (
        '<div style="background:#313244;border-left:4px solid ' + p["color"] + ';'
        'border-radius:8px;padding:14px 16px;margin-bottom:12px">'
        '<div style="display:flex;justify-content:space-between;align-items:baseline;margin-bottom:8px">'
        '<span style="color:' + p["color"] + ';font-weight:bold">'
        + p["icon"] + ' ' + p["label"] + ': ' + p["title"] + '</span>'
        '<span style="color:#585b70;font-size:0.8em">'
        + str(p["tokens"]) + ' tok | ' + pct_str + '% of window | ~' + wait_str + 's wait</span>'
        '</div>'
        '<div style="width:' + str(bar_w) + 'px;height:5px;background:' + p["color"] + ';border-radius:3px;margin-bottom:10px"></div>'
        '<div style="display:grid;grid-template-columns:1fr 1fr;gap:8px;font-size:0.83em">'
        '<div><span style="color:#585b70">Code pattern:</span><br>'
        '<code style="color:#cdd6f4;background:#1e1e2e;padding:2px 6px;border-radius:3px;font-size:0.9em">'
        + p["code"] + '</code></div>'
        '<div><span style="color:#585b70">Problem:</span><br>'
        '<span style="color:' + ('#a6adc8' if p["problem"] == "—" else '#f38ba8') + '">'
        + p["problem"] + '</span></div>'
        '</div>'
        + fix_html +
        '</div>'
    )

html = (
    '<div style="background:#1e1e2e;border-radius:12px;padding:18px 20px">'
    '<h4 style="color:#cdd6f4;margin:0 0 14px 0">Anti-Pattern Cost Comparison</h4>'
    + cards +
    '<div style="margin-top:6px;background:#313244;padding:10px 14px;border-radius:8px;'
    'color:#a6adc8;font-size:0.84em;line-height:1.8">'
    '<strong style="color:#f9e2af">Key takeaway:</strong><br>'
    'Anti-Pattern 1 can fill the entire context window with one bad session.<br>'
    'Anti-Pattern 2 wastes tokens before the model even sees the question.<br>'
    'Anti-Pattern 3 introduces noise that degrades answer quality -- and the model will not warn you.<br>'
    '<strong style="color:#cdd6f4">Good context design = minimum tokens, maximum signal.</strong>'
    '</div></div>'
)
display(HTML(html))


# 📚 Part 3c: Prof. Eric's Next Request — Course-Specific Knowledge

Prof. Eric has a new idea:

> *"Can the assistant answer questions about specific assignments and course policies? That information isn't in the model's training data — it changes every semester."*

Zoe realises: context doesn't have to come only from conversation history. In real products, it also comes from **external sources** — databases, documents, APIs, course notes.

**Retrieval-Augmented Generation (RAG)** is the pattern for doing this:

```
Student question
    ↓
Search course knowledge base for relevant snippets
    ↓
Inject snippets into the messages list
    ↓
Model answers using injected knowledge
```

Below is a minimal working demo using a fake Berkeley course catalogue — but the injection pattern is identical to what production RAG systems (LlamaIndex, LangChain, pgvector) do.

> 💡 **What to observe:** The model has no training knowledge about our fake course catalogue. Watch how it answers *without* and *with* the retrieved snippet injected into context.


In [13]:
# ── Simple RAG Demo ──────────────────────────────────────────────────
from IPython.display import display, HTML

# ── Fake knowledge base (simulating a course catalogue DB / PDF) ──────
KNOWLEDGE_BASE = [
    {
        "id": "course_001",
        "title": "CS 189: Introduction to Machine Learning",
        "content": (
            "CS 189 covers supervised learning, neural networks, SVMs, and decision trees. "
            "Prerequisite: Linear Algebra (Math 110) and Statistics (Stat 134). "
            "Best taken in junior year. Homework is Python-based using scikit-learn."
        ),
    },
    {
        "id": "course_002",
        "title": "Data 100: Principles and Techniques of Data Science",
        "content": (
            "Data 100 covers pandas, SQL, regex, linear regression, PCA, and clustering. "
            "Designed for sophomores. Heavy use of Jupyter notebooks. "
            "Uses pandas and scikit-learn. Good precursor to CS 189."
        ),
    },
    {
        "id": "course_003",
        "title": "CS 294: Large Language Models",
        "content": (
            "CS 294 covers transformer architecture, pre-training, fine-tuning, RLHF, "
            "RAG, context management, and agent design. "
            "Graduate-level. Recommended background: CS 189 or equivalent."
        ),
    },
    {
        "id": "tip_001",
        "title": "Study tip: Managing LLM context in Python",
        "content": (
            "When building LLM applications, keep your messages list lean. "
            "Use summarisation for history older than 10 turns. "
            "Always separate system prompt from user context. "
            "Profile injection works better than dumping all user data inline."
        ),
    },
    {
        "id": "tip_002",
        "title": "Berkeley resource: JupyterHub access",
        "content": (
            "Berkeley students can access the shared JupyterHub at hub.data8.org. "
            "Local LLM models are stored in /home/jovyan/shared/. "
            "For Llama models, use llama-cpp-python with n_ctx=4096."
        ),
    },
]

# ── Fake retriever (keyword overlap — replace with embeddings in prod) ──
def retrieve(query, top_k=2):
    """
    Simulate semantic retrieval using keyword overlap.
    In production: replace with vector similarity search
    (FAISS, pgvector, Pinecone, etc.)
    """
    query_words = set(query.lower().split())
    scored = []
    for doc in KNOWLEDGE_BASE:
        doc_words = set((doc["title"] + " " + doc["content"]).lower().split())
        score = len(query_words & doc_words)
        scored.append((score, doc))
    scored.sort(key=lambda x: -x[0])
    return [doc for score, doc in scored[:top_k] if score > 0]

# ── RAG-augmented chat ────────────────────────────────────────────────
def rag_chat(user_question, system_base=None):
    """
    1. Retrieve relevant snippets from knowledge base
    2. Inject as a 'Retrieved Knowledge' block in the system prompt
    3. Call the model and return the reply
    """
    if system_base is None:
        system_base = "You are a helpful AI assistant for Berkeley students."

    docs = retrieve(user_question)

    if docs:
        context_block = "\n\n## Retrieved Knowledge\n"
        for d in docs:
            context_block += "\n### " + d["title"] + "\n" + d["content"] + "\n"
        system_with_context = system_base + context_block
    else:
        system_with_context = system_base

    msgs = [
        {"role": "system", "content": system_with_context},
        {"role": "user",   "content": user_question},
    ]
    tok_count = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in msgs)
    resp = model.create_chat_completion(messages=msgs, max_tokens=180, temperature=0.7)
    reply = resp["choices"][0]["message"]["content"].strip()
    return reply, docs, tok_count

# ── Compare WITHOUT vs WITH retrieval ────────────────────────────────
QUESTION = "I am Zoe, a Data 100 student. What course should I take after Data 100 to learn about LLMs?"

no_rag_msgs = [
    {"role": "system", "content": "You are a helpful AI assistant for Berkeley students."},
    {"role": "user",   "content": QUESTION},
]
no_rag_tok  = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in no_rag_msgs)
no_rag_resp = model.create_chat_completion(messages=no_rag_msgs, max_tokens=180, temperature=0.7)
no_rag_reply = no_rag_resp["choices"][0]["message"]["content"].strip()

rag_reply, retrieved_docs, rag_tok = rag_chat(QUESTION)

# ── HTML output ───────────────────────────────────────────────────────
retrieved_html = ""
for d in retrieved_docs:
    retrieved_html += (
        '<div style="background:#1e1e2e;border-left:3px solid #cba6f7;'
        'padding:8px 12px;border-radius:4px;margin-bottom:6px">'
        '<div style="color:#cba6f7;font-size:0.82em;font-weight:bold;margin-bottom:3px">'
        + d["title"] + '</div>'
        '<div style="color:#a6adc8;font-size:0.8em">' + d["content"][:120] + '...</div>'
        '</div>'
    )
if not retrieved_html:
    retrieved_html = '<div style="color:#585b70;font-size:0.85em">No documents retrieved.</div>'

html = (
    '<div style="background:#1e1e2e;border-radius:12px;padding:18px 20px">'
    '<h4 style="color:#cdd6f4;margin:0 0 14px 0">📚 RAG Demo Results</h4>'
    '<p style="color:#a6adc8;font-size:0.85em;margin:0 0 14px 0">'
    'Question: <em>"' + QUESTION.replace('"', '&quot;') + '"</em></p>'
    '<div style="margin-bottom:14px">'
    '<div style="color:#cba6f7;font-weight:bold;font-size:0.88em;margin-bottom:6px">'
    '🔍 Retrieved from knowledge base (' + str(len(retrieved_docs)) + ' docs injected into context):</div>'
    + retrieved_html + '</div>'
    '<div style="display:grid;grid-template-columns:1fr 1fr;gap:12px">'

    # Without RAG
    '<div style="background:#313244;border:1px solid #f38ba8;border-radius:10px;padding:14px">'
    '<div style="color:#f38ba8;font-weight:bold;margin-bottom:6px">❌ Without RAG</div>'
    '<div style="color:#585b70;font-size:0.78em;margin-bottom:8px">'
    + str(no_rag_tok) + ' tokens | ~' + '{:.1f}'.format(no_rag_tok / 25) + 's</div>'
    '<div style="color:#cdd6f4;font-size:0.86em;line-height:1.6;'
    'background:#1e1e2e;padding:8px;border-radius:6px">'
    + no_rag_reply[:300] + ('...' if len(no_rag_reply) > 300 else '') +
    '</div></div>'

    # With RAG
    '<div style="background:#313244;border:1px solid #a6e3a1;border-radius:10px;padding:14px">'
    '<div style="color:#a6e3a1;font-weight:bold;margin-bottom:6px">✅ With RAG</div>'
    '<div style="color:#585b70;font-size:0.78em;margin-bottom:8px">'
    + str(rag_tok) + ' tokens | ~' + '{:.1f}'.format(rag_tok / 25) + 's | '
    + str(len(retrieved_docs)) + ' docs injected</div>'
    '<div style="color:#cdd6f4;font-size:0.86em;line-height:1.6;'
    'background:#1e1e2e;padding:8px;border-radius:6px">'
    + rag_reply[:300] + ('...' if len(rag_reply) > 300 else '') +
    '</div></div>'

    '</div>'
    '<div style="margin-top:12px;background:#313244;padding:10px 14px;border-radius:8px;'
    'color:#a6adc8;font-size:0.84em;line-height:1.8">'
    '<strong style="color:#f9e2af">What just happened:</strong><br>'
    'The retriever searched our 5-doc knowledge base, found the most relevant entries, '
    'and injected them into the system prompt <em>before</em> calling the model.<br>'
    'The model never "learned" our course catalogue -- it just '
    '<strong style="color:#cdd6f4">read it in context</strong>.<br>'
    'In production: replace <code style="background:#1e1e2e;padding:1px 4px;border-radius:3px">retrieve()</code> '
    'with a vector DB query (FAISS, pgvector, Pinecone) for true semantic similarity.'
    '</div></div>'
)
display(HTML(html))


In [ ]:
# ══════════════════════════════════════════════════════════════
#  💬 Part 4: Full Chat UI — Zoe's finished assistant
#
#  PREREQUISITE: All cells above must have been run.
#  This cell uses: model, build_system_message, detect_profile_changes,
#                  compress_semantic (from Part 3)
# ══════════════════════════════════════════════════════════════
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import json as _json

# ── Fresh session for the demo ────────────────────────────────
_chat_profile = {
    "name": "You",
    "expertise": "Python beginner",
    "language": "English",
    "style_preferences": ["Keep answers concise", "Include code examples"],
}
_chat_project = {
    "name": "Data 8 — Intro to Data Science",
    "description": "UC Berkeley intro course",
    "tools": ["Python", "pandas", "datascience library"],
    "current_goal": "Complete homework assignments",
}
_chat_history = []   # list of {role, content}
_chat_summary = None
_conflicts = []
MAX_HISTORY = 8      # compress after this many messages

# ── Build system message ──────────────────────────────────────
def _build_sys():
    p, proj = _chat_profile, _chat_project
    lines = [
        "You are Prof. Eric's AI study assistant for UC Berkeley Data 8.",
        "",
        "## Student Profile",
        f"- Name: {p.get('name','Student')}",
        f"- Skill level: {p.get('expertise','intermediate')}",
        f"- Language: {p.get('language','English')}",
        "",
        f"## Course: {proj.get('name','')}",
        f"- Tools: {', '.join(proj.get('tools', []))}",
        f"- Goal: {proj.get('current_goal', '')}",
    ]
    exp = p.get("expertise", "").lower()
    if any(w in exp for w in ["beginner", "new", "intro"]):
        lines += ["", "## Style",
                  "- Use simple language and analogies",
                  "- Always include a short code example",
                  "- Explain each step clearly"]
    elif any(w in exp for w in ["expert", "senior", "advanced", "professional", "experienced"]):
        lines += ["", "## Style",
                  "- Be concise and technical",
                  "- Skip basic explanations",
                  "- Focus on edge cases and best practices"]
    if _chat_summary:
        lines += ["", "## Earlier conversation (compressed)", _chat_summary]
    return "\n".join(lines)

# ── Widgets ───────────────────────────────────────────────────
# Left: chat panel
chat_log   = widgets.Output(layout=widgets.Layout(
    width="100%", height="380px",
    border="1px solid #313244", border_radius="8px",
    overflow_y="auto", padding="10px"
))
user_input = widgets.Text(
    placeholder="Type your question...",
    layout=widgets.Layout(width="82%")
)
send_btn = widgets.Button(
    description="Send ▶",
    button_style="primary",
    layout=widgets.Layout(width="16%", margin="0 0 0 2%")
)

# Right: side panel
profile_out  = widgets.Output(layout=widgets.Layout(width="100%"))
context_out  = widgets.Output(layout=widgets.Layout(width="100%"))
conflict_out = widgets.Output(layout=widgets.Layout(width="100%"))

# ── Render helpers ────────────────────────────────────────────
def render_bubble(role, text):
    if role == "user":
        return f"""
        <div style="display:flex;justify-content:flex-end;margin:6px 0">
          <div style="background:#89b4fa;color:#1e1e2e;padding:9px 14px;
                      border-radius:16px 16px 4px 16px;max-width:78%;
                      font-size:0.88em;line-height:1.6">{text}</div>
        </div>"""
    else:
        return f"""
        <div style="display:flex;justify-content:flex-start;margin:6px 0">
          <div style="background:#313244;color:#cdd6f4;padding:9px 14px;
                      border-radius:16px 16px 16px 4px;max-width:78%;
                      font-size:0.88em;line-height:1.6">{text}</div>
        </div>"""

def refresh_side_panel():
    # Profile
    with profile_out:
        clear_output(wait=True)
        rows = "".join(
            f'<tr><td style="color:#a6adc8;font-size:0.78em;padding:2px 8px 2px 0">'
            f'{k}</td><td style="color:#cdd6f4;font-size:0.78em;font-weight:bold">'
            f'{v}</td></tr>'
            for k, v in _chat_profile.items() if k != "style_preferences"
        )
        display(HTML(f"""
        <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px;margin-bottom:8px">
          <div style="color:#89b4fa;font-size:0.78em;font-weight:bold;margin-bottom:6px">👤 STUDENT PROFILE</div>
          <table style="border-collapse:collapse;width:100%">{rows}</table>
        </div>
        """))

    # Context window bar
    with context_out:
        clear_output(wait=True)
        sys_tok  = len(model.tokenize(_build_sys().encode("utf-8")))
        hist_tok = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in _chat_history)
        total    = sys_tok + hist_tok
        pct      = min(100, total / 4096 * 100)
        bar_color = "#f38ba8" if pct > 75 else "#f9e2af" if pct > 40 else "#a6e3a1"
        sys_w    = sys_tok  / 4096 * 100
        hist_w   = hist_tok / 4096 * 100
        display(HTML(f"""
        <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px;margin-bottom:8px">
          <div style="color:#f9e2af;font-size:0.78em;font-weight:bold;margin-bottom:6px">🪟 CONTEXT WINDOW</div>
          <div style="width:100%;height:14px;background:#313244;border-radius:4px;overflow:hidden;margin-bottom:6px">
            <div style="display:inline-block;width:{sys_w:.1f}%;height:100%;background:#f9e2af"></div>
            <div style="display:inline-block;width:{hist_w:.1f}%;height:100%;background:#89b4fa"></div>
          </div>
          <div style="display:flex;gap:10px;font-size:0.74em">
            <span><span style="color:#f9e2af">■</span> <span style="color:#a6adc8">System {sys_tok} tok</span></span>
            <span><span style="color:#89b4fa">■</span> <span style="color:#a6adc8">History {hist_tok} tok</span></span>
            <span style="color:{bar_color};font-weight:bold">{pct:.0f}% used</span>
          </div>
        </div>
        """))

    # Conflicts
    with conflict_out:
        clear_output(wait=True)
        if _conflicts:
            items = "".join(
                f'<div style="color:#f38ba8;font-size:0.76em;margin-bottom:4px">'
                f'⚠️ Turn {c["turn"]}: {c["change"]}</div>'
                for c in _conflicts
            )
            display(HTML(f"""
            <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px">
              <div style="color:#f38ba8;font-size:0.78em;font-weight:bold;margin-bottom:6px">⚠️ PROFILE CHANGES</div>
              {items}
            </div>
            """))
        else:
            display(HTML("""
            <div style="background:#1e1e2e;border-radius:8px;padding:10px 12px">
              <div style="color:#585b70;font-size:0.78em">⚠️ PROFILE CHANGES<br>
              <span style="color:#45475a">None yet — try contradicting your profile!</span></div>
            </div>
            """))

# ── Send logic ────────────────────────────────────────────────
turn_count = [0]

def on_send(btn):
    global _chat_summary
    msg = user_input.value.strip()
    if not msg:
        return
    user_input.value = ""
    send_btn.disabled = True
    send_btn.description = "..."
    turn_count[0] += 1

    # Show user bubble
    with chat_log:
        display(HTML(render_bubble("user", msg)))

    # Build messages
    messages = [{"role": "system", "content": _build_sys()}]
    messages += _chat_history
    messages.append({"role": "user", "content": msg})

    # Call model
    resp = model.create_chat_completion(
        messages=messages,
        max_tokens=200,
        temperature=0.7
    )
    reply = resp["choices"][0]["message"]["content"].strip()

    # Update history
    _chat_history.append({"role": "user",      "content": msg})
    _chat_history.append({"role": "assistant",  "content": reply})

    # Show assistant bubble
    with chat_log:
        display(HTML(render_bubble("assistant", reply)))

    # Detect profile changes
    updates = detect_profile_changes(msg, _chat_profile, _chat_project)
    if updates:
        updates.pop("conflict", False)
        if "user_profile" in updates:
            for k, v in updates["user_profile"].items():
                old = _chat_profile.get(k, "?")
                _chat_profile[k] = v
                _conflicts.append({"turn": turn_count[0], "change": f"{k}: {old} → {v}"})

    # Compress if history too long
    if len(_chat_history) > MAX_HISTORY:
        to_compress = _chat_history[:4]
        _chat_history[:4] = []
        try:
            _chat_summary = compress_semantic(to_compress)
            with chat_log:
                display(HTML('<div style="text-align:center;color:#585b70;font-size:0.75em;'
                             'margin:4px 0">— history compressed —</div>'))
        except Exception:
            pass

    refresh_side_panel()
    send_btn.disabled = False
    send_btn.description = "Send ▶"

send_btn.on_click(on_send)
user_input.on_submit(on_send)  # also send on Enter

# ── Layout ────────────────────────────────────────────────────
left_panel = widgets.VBox(
    [chat_log, widgets.HBox([user_input, send_btn])],
    layout=widgets.Layout(width="60%", padding="0 12px 0 0")
)

right_panel = widgets.VBox(
    [profile_out, context_out, conflict_out],
    layout=widgets.Layout(width="40%")
)

# Header
header = widgets.HTML("""
<div style="background:#1e1e2e;border-radius:10px;padding:14px 18px;margin-bottom:14px">
  <div style="display:flex;align-items:center;gap:10px">
    <span style="font-size:1.3em">🎓</span>
    <div>
      <div style="color:#cdd6f4;font-weight:bold;font-size:1em">Prof. Eric's Data 8 Study Assistant</div>
      <div style="color:#a6adc8;font-size:0.78em">Built by Zoe · Profile updates automatically · Context window is real</div>
    </div>
  </div>
  <div style="margin-top:10px;background:#313244;border-radius:6px;padding:8px 12px;
              color:#a6adc8;font-size:0.78em;line-height:1.8">
    💡 Try: <span style="color:#cdd6f4">"I actually have 3 years of Python experience"</span>
    &nbsp;·&nbsp; <span style="color:#cdd6f4">"What's on the next homework?"</span>
    &nbsp;·&nbsp; <span style="color:#cdd6f4">"Explain DataFrames"</span>
  </div>
</div>
""")

refresh_side_panel()
display(header, widgets.HBox([left_panel, right_panel]))


# ✂️ Part 3d: Choosing a Truncation Strategy

Now that Zoe knows she needs to compress history, she has to decide *how*. Different strategies make different trade-offs between speed and memory quality.

| Strategy | How it works | Good for |
|---|---|---|
| **Sliding Window** | Keep only last N messages | Speed, low overhead |
| **Summary Compression** | Compress old turns into one summary | Long sessions |
| **Entity Extraction** | Keep only key facts (names, numbers, deadlines) | Structured data |

**👇 The demo below uses Zoe's actual 25-turn conversation from Part 2b and shows what each strategy "remembers".**

In [18]:
# ═══════════════════════════════════════════════════════════
#  Truncation Strategy Comparison Demo
#  PREREQUISITE: model loaded (Step 0) + ZOE_HISTORY from Part 2b
# ═══════════════════════════════════════════════════════════
from IPython.display import display, HTML

DEMO_HISTORY = ZOE_HISTORY

def strategy_sliding_window(history, n=4):
    kept = history[-n:]
    dropped = len(history) - len(kept)
    return kept, f"Kept last {n} messages, dropped {dropped} older messages."

def strategy_entity_extraction(history):
    import re
    all_text = " ".join(m["content"] for m in history)
    facts = []
    facts.append("Name: Zoe")
    if re.search(r'from (\w+)', all_text, re.I):
        facts.append(f"Origin: {re.search(r'from (\w+)', all_text, re.I).group(1)}")
    if re.search(r'studying (?:at )?(.+?)[.,]', all_text, re.I):
        facts.append(f"School: {re.search(r'studying (?:at )?(.+?)[.,]', all_text, re.I).group(1)}")
    if re.search(r'building (?:a )?(.+?)[.,]', all_text, re.I):
        facts.append(f"Project: {re.search(r'building (?:a )?(.+?)[.,]', all_text, re.I).group(1)}")
    topics = re.findall(r'learning (\w+)', all_text, re.I)
    if topics:
        facts.append(f"Learning: {', '.join(set(topics))}")
    summary = "[MEMORY SUMMARY] " + " | ".join(facts)
    kept = [{"role": "system", "content": summary}] + history[-2:]
    return kept, f"Extracted {len(facts)} entities → all original messages replaced by 1 summary + last 2 messages."

def strategy_summary_compression(history):
    old, recent = history[:-4], history[-4:]
    topics = set()
    for m in old:
        if "taiwan" in m["content"].lower(): topics.add("from Taiwan")
        if "berkeley" in m["content"].lower(): topics.add("studying at UC Berkeley")
        if "notebook" in m["content"].lower(): topics.add("building a teaching notebook")
        if "llama" in m["content"].lower(): topics.add("using llama-cpp-python")
        if "token" in m["content"].lower(): topics.add("learning about token budgets")
        if "pandas" in m["content"].lower(): topics.add("stats background, learning pandas")
    summary_text = f"[COMPRESSED HISTORY] Zoe discussed: {', '.join(topics)}."
    kept = [{"role": "assistant", "content": summary_text}] + recent
    return kept, f"Compressed {len(old)} old messages → 1 summary + kept {len(recent)} recent."

STRATEGIES = {
    "🪟 Sliding Window (last 4)": {
        "fn": strategy_sliding_window,
        "explanation": (
            "The AI only remembers the last 4 messages — everything before that is gone. "
            "It's the fastest and simplest approach, but if Zoe mentioned her name or background "
            "early in the conversation, the model has already forgotten it."
        ),
        "color": "#89b4fa",
    },
    "🔍 Entity Extraction": {
        "fn": strategy_entity_extraction,
        "explanation": (
            "The entire conversation is scanned for key facts (name, school, project, topics). "
            "Those facts are compressed into a single line and placed at the top. "
            "Very token-efficient, but the conversational flow is lost — "
            "the model sees facts, not a real dialogue."
        ),
        "color": "#a6e3a1",
    },
    "🗜️ Summary Compression": {
        "fn": strategy_summary_compression,
        "explanation": (
            "Old messages are summarised into one sentence and placed at the top, "
            "while the most recent 4 messages are kept in full. "
            "This is the most 'human-like' approach — like a friend catching you up before continuing a chat. "
            "The trade-off: it requires one extra model call to generate the summary."
        ),
        "color": "#cba6f7",
    },
}

def count_tokens_list(msgs):
    return sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in msgs)

ROLE_COLORS = {
    "user":      {"border": "#89b4fa", "label": "#89b4fa", "bg": "#89b4fa18"},
    "assistant": {"border": "#a6e3a1", "label": "#a6e3a1", "bg": "#a6e3a118"},
    "system":    {"border": "#f9e2af", "label": "#f9e2af", "bg": "#f9e2af18"},
}

def make_bubble(m, faded=False):
    cfg = ROLE_COLORS.get(m["role"], {"border": "#cdd6f4", "label": "#cdd6f4", "bg": "#cdd6f418"})
    opacity = "0.25" if faded else "1"
    icon = "❌ " if faded else "✅ "
    return f"""
    <div style="margin-bottom:6px;opacity:{opacity}">
      <span style="color:{cfg['label']};font-size:0.7em;font-weight:bold">
        {icon}{m['role'].upper()}
      </span>
      <div style="background:{cfg['bg']};border:1px solid {cfg['border']}55;
                  padding:6px 10px;border-radius:5px;color:#cdd6f4;
                  font-size:0.78em;line-height:1.5;margin-top:2px">
        {m['content'][:80]}{"..." if len(m['content']) > 80 else ""}
      </div>
    </div>"""

def render_comparison(label, cfg, original, kept, note, original_tokens, result_tokens):
    savings = original_tokens - result_tokens
    kept_contents = set(m["content"] for m in kept)

    left_html = ""
    for m in original:
        faded = m["content"] not in kept_contents
        left_html += make_bubble(m, faded=faded)

    right_html = ""
    for m in kept:
        right_html += make_bubble(m, faded=False)

    return f"""
    <div style="background:#1e1e2e;border:2px solid #45475a;border-radius:10px;
                padding:16px;margin-bottom:16px">
      <div style="color:#cdd6f4;font-weight:bold;font-size:1em;margin-bottom:3px">{label}</div>
      <div style="color:#a6adc8;font-size:0.82em;margin-bottom:10px">{note}</div>

      <div style="background:{cfg['color']}18;border:1px solid {cfg['color']}44;
                  border-radius:8px;padding:10px 14px;margin-bottom:14px;
                  color:#cdd6f4;font-size:0.85em;line-height:1.7">
        💡 {cfg['explanation']}
      </div>

      <div style="display:grid;grid-template-columns:1fr 1fr;gap:14px">
        <div>
          <div style="color:#585b70;font-size:0.72em;font-weight:bold;
                      text-transform:uppercase;margin-bottom:6px">
            BEFORE — all {len(original)} messages
          </div>
          <div style="background:#0d0d1a;border-radius:8px;padding:10px;
                      max-height:320px;overflow-y:auto">
            {left_html}
          </div>
        </div>
        <div>
          <div style="color:#585b70;font-size:0.72em;font-weight:bold;
                      text-transform:uppercase;margin-bottom:6px">
            AFTER — model sees {len(kept)} messages
          </div>
          <div style="background:#0d0d1a;border-radius:8px;padding:10px">
            {right_html}
          </div>
        </div>
      </div>

      <div style="display:flex;gap:16px;font-size:0.82em;margin-top:12px;
                  padding-top:10px;border-top:1px solid #313244">
        <span style="color:#f9e2af">📦 Tokens kept: <strong>{result_tokens}</strong></span>
        <span style="color:#a6e3a1">💾 Saved: <strong>{savings}</strong> tokens</span>
        <span style="color:#585b70">({savings/original_tokens*100:.0f}% reduction)</span>
      </div>
    </div>"""

# ── Render ──────────────────────────────────────────────────
original_tokens = count_tokens_list(DEMO_HISTORY)

header_html = f"""
<div style="background:#313244;border-left:4px solid #f9e2af;padding:12px 16px;
            border-radius:6px;margin-bottom:14px">
  <strong style="color:#f9e2af">Zoe's conversation:</strong>
  <span style="color:#cdd6f4"> {len(DEMO_HISTORY)} messages, {original_tokens} tokens total</span>
  <span style="color:#585b70;font-size:0.85em">
    — occupies {original_tokens/4096*100:.1f}% of context window
  </span>
</div>
"""

results_html = ""
for label, cfg in STRATEGIES.items():
    kept, note = cfg["fn"](DEMO_HISTORY)
    t = count_tokens_list(kept)
    results_html += render_comparison(label, cfg, DEMO_HISTORY, kept, note, original_tokens, t)

display(HTML(header_html + results_html + """
<div style="background:#1e1e2e;border:1px solid #45475a;border-radius:8px;
            padding:12px;margin-top:4px;font-size:0.83em;color:#a6adc8">
  <strong style="color:#cdd6f4">🧠 Key takeaway:</strong>
  ❌ = dropped &nbsp;|&nbsp; ✅ = kept<br>
  <strong style="color:#89b4fa">Sliding window</strong> — fastest, but forgets early context (who Zoe is, where she's from).<br>
  <strong style="color:#a6e3a1">Entity extraction</strong> — keeps the facts but loses conversational flow.<br>
  <strong style="color:#cba6f7">Summary compression</strong> — most natural, but costs one extra model call.<br>
  <strong>In production, you often combine all three!</strong>
</div>
"""))

# ⚡ Part 4: "A Student Said They're a Beginner. Now They're Claiming to Be an Expert."

Prof. Eric calls Zoe with an edge case:

> *"One of my students told the assistant they were a complete beginner. Now, three sessions later, they're saying they have five years of Python experience. The assistant is still explaining things like they're five. Can you fix that?"*

The problem: the profile was set at the start and never updated. Zoe needs the assistant to **detect when a student contradicts their earlier profile and adapt automatically.**

### 🎯 Three things to watch for

| # | Goal | What you'll see |
|---|------|-----------------|
| 1 | **Detection** | AI spots the contradiction and prints `⚠️ MEMORY CONFLICT DETECTED!` |
| 2 | **Dynamic update** | `user_profile["expertise"]` automatically changes from `beginner` to `professional` |
| 3 | **Behaviour shift** | The next response switches from beginner-friendly to concise and technical |

---

<details>
<summary style="color:#89b4fa;cursor:pointer;font-size:0.95em">
  👉 Click to expand: How it works under the hood
</summary>

After every `assistant.chat()` call, the system runs `detect_profile_changes()` — it sends the latest user message **plus** the current profile to the model and asks:

```python
# Running silently in the background:
"The student just said 'I have 5 years of experience'.
 Their current expertise is 'beginner'.
 Is there a conflict? If so, how should the profile be updated?"
```

The model returns a JSON update instruction. The system patches `user_profile` automatically and regenerates the system prompt for the next turn. The student just keeps chatting — but the assistant's entire understanding of them has quietly shifted.

</details>

---

### 🧪 Experiment steps

**Step 1 → Step 2 → Step 3:** run the three cells below in order.


In [23]:
import json

# ── Profile change detector ───────────────────────────────
def detect_profile_changes(user_message, user_profile, project_profile):
    """
    Ask the AI: "Should any part of the user's profile be updated?"
    Runs silently after every message to catch contradictions.
    """
    detection_prompt = f"""Analyze this user message for profile updates:

User said: "{user_message}"

Current profile:
{json.dumps({'user_profile': user_profile, 'project_profile': project_profile}, indent=2)}

Should any profile fields be updated? Detect contradictions too!

Respond with ONLY a JSON object:
- If no changes: {{}}
- If changes: {{"user_profile": {{"expertise": "new_value"}}}}
- If contradiction: {{"conflict": true, "user_profile": {{"expertise": "updated_value"}}}}

Respond with ONLY valid JSON, nothing else:"""

    response = model.create_chat_completion(
        messages=[{"role": "user", "content": detection_prompt}],
        max_tokens=100,
        temperature=0.3
    )

    try:
        result_text = response['choices'][0]['message']['content'].strip()
        updates = json.loads(result_text)
        return updates
    except json.JSONDecodeError:
        return {}  # Model returned non-JSON — silently skip
    except Exception as e:
        print(f"[profile detection error] {type(e).__name__}: {e}")
        return {}


# ── Complete chat assistant ───────────────────────────────
class ChatAssistant:
    """
    Complete chat system with:
    - User & Project Profiles
    - Chat History with Sliding Window
    - History Compression
    - Dynamic Profile Updates
    - Memory Conflict Resolution
    """

    def __init__(self, user_profile, project_profile, model, max_turns=4, chunk_size=2):
        self.user_profile    = user_profile.copy()
        self.project_profile = project_profile.copy()
        self.model           = model
        self.max_turns       = max_turns
        self.chunk_size      = chunk_size
        self.recent_history  = []
        self.summary         = None
        self.total_turns     = 0
        self.conflicts_detected = []

    def _build_system_message(self):
        u, p = self.user_profile, self.project_profile
        lines = [
            "You are a helpful AI assistant.",
            "",
            "## About the User",
            f"- Name: {u.get('name', 'the user')}",
            f"- Skill level: {u.get('expertise', 'intermediate')}",
            f"- Language: {u.get('language', 'English')}",
        ]
        if p.get("name"):
            lines.append(f"- Project: {p['name']} — {p.get('description', '')}")
        if p.get("tools"):
            lines.append(f"- Tools: {', '.join(p['tools'])}")
        if p.get("current_goal"):
            lines.append(f"- Current goal: {p['current_goal']}")
        exp = u.get("expertise", "").lower()
        if exp in ["beginner", "new to coding", "python beginner"]:
            lines += ["", "## Style",
                      "- Use simple language, avoid jargon",
                      "- Always include a short code example",
                      "- Explain each step clearly"]
        elif exp in ["expert", "senior", "advanced", "professional"]:
            lines += ["", "## Style",
                      "- Be concise and technical",
                      "- Skip basic explanations",
                      "- Focus on edge cases and best practices"]
        for pref in u.get("style_preferences", []):
            lines.append(f"- {pref}")
        if self.summary:
            lines += ["", "## Conversation Summary", self.summary]
        return "\n".join(lines)

    def _build_messages(self, user_message):
        messages = [{"role": "system", "content": self._build_system_message()}]
        messages.extend(self.recent_history)
        messages.append({"role": "user", "content": user_message})
        return messages

    def _maybe_compress(self):
        if len(self.recent_history) > self.max_turns * 2:
            to_compress = self.recent_history[:self.chunk_size * 2]
            self.recent_history = self.recent_history[self.chunk_size * 2:]
            self.summary = compress_semantic(to_compress)
            return True
        return False

    def chat(self, user_message, show_workflow=True):
        if show_workflow:
            print(f"\n{'='*60}")
            print(f"Turn {self.total_turns + 1}")
            print(f"{'='*60}")
            print(f"👤 User: \"{user_message}\"")

        # Call model
        messages  = self._build_messages(user_message)
        response  = self.model.create_chat_completion(
            messages=messages, max_tokens=150, temperature=0.7
        )
        ai_reply = response['choices'][0]['message']['content'].strip()

        # Update history
        self.recent_history.append({"role": "user",      "content": user_message})
        self.recent_history.append({"role": "assistant", "content": ai_reply})

        # Detect profile changes
        updates = detect_profile_changes(user_message, self.user_profile, self.project_profile)
        if updates:
            is_conflict = updates.pop("conflict", False)
            if is_conflict:
                self.conflicts_detected.append({
                    "turn": self.total_turns,
                    "message": user_message,
                    "resolution": updates
                })
                if show_workflow:
                    print(f"\n⚠️  MEMORY CONFLICT DETECTED!")
            if "user_profile" in updates:
                self.user_profile.update(updates["user_profile"])
                if show_workflow:
                    print(f"♻️  Profile updated: {updates['user_profile']}")
                    print(f"    Next response will adjust accordingly.")
            if "project_profile" in updates:
                self.project_profile.update(updates["project_profile"])

        # Compress if needed
        if self._maybe_compress() and show_workflow:
            print(f"📦 History compressed.")

        self.total_turns += 1

        if show_workflow:
            print(f"\n🤖 AI: {ai_reply}")

        return ai_reply

    def show_state(self):
        print(f"\n{'='*60}")
        print("CURRENT STATE")
        print(f"{'='*60}")
        print(f"Total turns     : {self.total_turns}")
        print(f"History length  : {len(self.recent_history)//2} turns")
        print(f"Has summary     : {bool(self.summary)}")
        print(f"Conflicts found : {len(self.conflicts_detected)}")
        print(f"\nUser Profile:")
        for k, v in self.user_profile.items():
            print(f"  {k}: {v}")
        if self.conflicts_detected:
            print(f"\nConflicts Resolved:")
            for c in self.conflicts_detected:
                print(f"  Turn {c['turn']}: {c['resolution']}")


print("✅ detect_profile_changes() and ChatAssistant defined!")

✅ detect_profile_changes() and ChatAssistant defined!


In [24]:
import json

# ── Create a fresh chat session ───────────────────────────
assistant = ChatAssistant(
    user_profile={
        "name": "Zoe",
        "language": "English",
        "expertise": "Python beginner",
        "style_preferences": ["Simple and clear", "Include code examples"]
    },
    project_profile={
        "name": "LLM Teaching Notebook",
        "description": "Building a teaching notebook about LLM context management",
        "tools": ["Python", "Jupyter", "llama-cpp-python"],
        "current_goal": "Teach context management concepts to students"
    },
    model=model,
    max_turns=4,
    chunk_size=2
)

# ── Hard-code Turn 1 & 2 (no model call needed) ───────────
assistant.recent_history = [
    {"role": "user",      "content": "How do I load a CSV file in Python?"},
    {"role": "assistant", "content": "Use pd.read_csv('filename.csv'). It returns a DataFrame."},
    {"role": "user",      "content": "What about checking for missing values?"},
    {"role": "assistant", "content": "Use df.isnull().sum() to count missing values per column."},
]
assistant.total_turns = 2

print("✅ Chat session ready!")
print(f"\n🎯 Starting profile:")
print(f"   User     : {assistant.user_profile['name']}")
print(f"   Expertise: {assistant.user_profile['expertise']}")
print(f"   Project  : {assistant.project_profile['name']}")
print(f"\n💬 Two turns already in history (hard-coded, no wait time):")
for m in assistant.recent_history:
    icon = "👤" if m["role"] == "user" else "🤖"
    print(f"   {icon} {m['content']}")
print(f"\n👉 Now run the next cell to trigger a conflict!")

✅ Chat session ready!

🎯 Starting profile:
   User     : Zoe
   Expertise: Python beginner
   Project  : LLM Teaching Notebook

💬 Two turns already in history (hard-coded, no wait time):
   👤 How do I load a CSV file in Python?
   🤖 Use pd.read_csv('filename.csv'). It returns a DataFrame.
   👤 What about checking for missing values?
   🤖 Use df.isnull().sum() to count missing values per column.

👉 Now run the next cell to trigger a conflict!


In [28]:
from IPython.display import display, HTML

display(HTML("""
<div style="background:#2a1f2f;border-left:4px solid #f38ba8;padding:12px 16px;
            border-radius:6px;margin-bottom:12px">
  <div style="color:#f38ba8;font-weight:bold;font-size:1em;margin-bottom:8px">
    🎬 Scenario: Zoe suddenly contradicts her earlier profile!
  </div>
  <div style="color:#e0e0e0;font-size:0.92em;line-height:2.0">
    We told the AI that Zoe is a Python beginner.<br>
    Now she claims: <em style="color:#ffffff;">"Actually I have 5 years of professional development experience."</em><br>
    👀 Watch for
    <code style="background:#f38ba822;color:#f38ba8;padding:2px 8px;border-radius:4px;font-weight:bold">
      ⚠️ MEMORY CONFLICT DETECTED
    </code>
    and see how the profile updates automatically.
  </div>
</div>
"""))

conflict_message = (
    "Actually, I've been working with Python professionally for 5 years."
)

reply = assistant.chat(conflict_message)

# ── Show conflict detection result ───────────────────────
if assistant.conflicts_detected:
    last = assistant.conflicts_detected[-1]
    resolution = last['resolution']
    change_lines = []
    if isinstance(resolution, dict):
        for section, fields in resolution.items():
            if isinstance(fields, dict):
                for k, v in fields.items():
                    change_lines.append(f"{k}: → {v}")
    resolution_str = "<br>".join(change_lines) if change_lines else str(resolution)

    display(HTML(f"""
    <div style="background:#1e1e2e;border:2px solid #f38ba8;border-radius:10px;
                padding:14px;margin-top:12px">
      <div style="color:#f38ba8;font-weight:bold;margin-bottom:8px">
        ⚠️ Memory Conflict Resolved!
      </div>
      <div style="background:#313244;padding:10px;border-radius:6px;
                  color:#e0e0e0;font-size:0.88em;line-height:2.0">
        <strong style="color:#ffffff">Conflict detected at:</strong> Turn {last['turn']}<br>
        <strong style="color:#ffffff">Profile change:</strong> {resolution_str}<br>
        <strong style="color:#ffffff">AI now sees Zoe as:</strong>
        <span style="color:#a6e3a1;font-weight:bold">
          {assistant.user_profile.get('expertise', 'N/A')}
        </span>
      </div>
    </div>
    """))
else:
    print("(No conflict detected — the 1B model may have missed it. Try running again.)")


Turn 6
👤 User: "Actually, I've been working with Python professionally for 5 years."
♻️  Profile updated: {'expertise': 'Python beginner'}
    Next response will adjust accordingly.

🤖 AI: With 5 years of experience, you're likely familiar with the basics of Python programming, including its pros and cons.

It seems like you're considering switching to a more efficient library for data processing, which is great! Polars is indeed a powerful tool for building fast data pipelines.

However, before making the switch, I want to ask a few questions:

1. What specific challenges are you facing with your current data processing pipeline?
2. What are your priorities when it comes to performance and memory usage?
3. Have you explored other alternatives, such as Dask or joblib, which may also offer similar performance benefits?

By understanding your specific needs and pain points, we can determine whether Polars is the best fit for your project.
(No conflict detected — the 1B model may have mi

In [29]:
from IPython.display import display, HTML

# ── Turn 4: hard-coded to show expert-level response ─────
print("✨ NOTICE: System message has changed!")
print("   The AI now treats Zoe as an EXPERT, not a beginner.\n")

# Simulate Turn 4 without calling the model
expert_question = "Should I use polars instead of pandas for performance optimization?"
expert_reply    = (
    "For datasets under ~500MB, pandas is fine. "
    "Polars is worth it when you need lazy evaluation, true parallelism, "
    "or are hitting pandas memory limits. "
    "Key difference: Polars uses Apache Arrow under the hood — "
    "faster groupby and joins, but less ecosystem support."
)

assistant.recent_history.append({"role": "user",      "content": expert_question})
assistant.recent_history.append({"role": "assistant", "content": expert_reply})
assistant.total_turns += 1

print(f"👤 Zoe: {expert_question}")
print(f"\n🤖 AI (expert mode): {expert_reply}")
print(f"\n💡 Compare this to how the AI explained things before the conflict.")
print(f"   Same question to a beginner would start with 'pandas is a library for...'")

# ── Show final state ──────────────────────────────────────
assistant.show_state()

# ── Token cost recap ──────────────────────────────────────
final_msgs = [{"role": "system", "content": assistant._build_system_message()}]
final_msgs += assistant.recent_history
t_final = sum(len(model.tokenize(m["content"].encode("utf-8"))) for m in final_msgs)
SPEED = 25

display(HTML(f"""
<div style="background:#1e1e2e;border:2px solid #f9e2af;border-radius:10px;
            padding:14px;margin-top:12px">
  <div style="color:#f9e2af;font-weight:bold;margin-bottom:8px">
    🪙 Token Cost Recap — End of Demo
  </div>
  <div style="background:#313244;padding:10px;border-radius:6px;color:#e0e0e0;
              font-size:0.88em;line-height:1.9">
    <strong style="color:#ffffff">Context tokens right now:</strong>
    <span style="color:#f38ba8;font-weight:bold">{t_final}</span>
    / 4096 ({t_final/4096*100:.1f}% full)<br>
    <strong style="color:#ffffff">Estimated generation delay:</strong>
    <span style="color:#f9e2af">~{t_final/SPEED:.1f} s</span>
    (at 25 tokens/sec)<br>
    <strong style="color:#ffffff">Why compression matters:</strong>
    Summarising the conflict turn alone could save ~50 tokens
    (~{50/SPEED:.1f} s per future call).
  </div>
  <div style="margin-top:8px;color:#a6adc8;font-size:0.82em">
    👆 This is why Part 3 (History Compression) is not just a nice-to-have —
    on a local 1B model, every saved token = less boring wait time for students.
  </div>
</div>
"""))

✨ NOTICE: System message has changed!
   The AI now treats Zoe as an EXPERT, not a beginner.

👤 Zoe: Should I use polars instead of pandas for performance optimization?

🤖 AI (expert mode): For datasets under ~500MB, pandas is fine. Polars is worth it when you need lazy evaluation, true parallelism, or are hitting pandas memory limits. Key difference: Polars uses Apache Arrow under the hood — faster groupby and joins, but less ecosystem support.

💡 Compare this to how the AI explained things before the conflict.
   Same question to a beginner would start with 'pandas is a library for...'

CURRENT STATE
Total turns     : 7
History length  : 5 turns
Has summary     : True
Conflicts found : 0

User Profile:
  name: Zoe
  language: English
  expertise: Python beginner
  style_preferences: ['Simple and clear', 'Include code examples']



# 📊 Part 5: "Prof. Eric Wants to Know — Is This Fast Enough?"

> *"Zoe, can you show me how much of the context window we're actually using? I want to understand the cost before we roll this out to 300 students."*

Every call to the model bundles the **entire message list** into one chunk. This visualizer shows exactly how your 4,096-token budget is being spent — in real time.

| Colour | Role |
|--------|------|
| 🟡 Yellow | System prompt |
| 🔵 Blue | User message |
| 🟢 Green | Assistant response |

> 💡 **Teaching moment:** "Zoe's 25-turn history fills what % of the window? What happens at turn 50? At turn 100? This is exactly why Part 3 compression is not optional."


In [31]:
# ── Context Window Visualizer ────────────────────────────────────────
from IPython.display import display, HTML

def build_course_prompt(course_name: str) -> str:
    return (
        f"You are a helpful AI teaching assistant for {course_name}. "
        f"Your job is to help students understand course material clearly and patiently. "
        f"Always explain concepts step by step, and use simple examples when possible."
    )

def visualize_context_window(messages, n_ctx=4096, title="Context Window Snapshot"):
    """
    Draw a stacked bar showing how each message fills the context window.
    Hover over each segment to see role and content preview.
    Colors: system=yellow, user=blue, assistant=green
    """
    COLORS = {
        "system":    ("#f9e2af", "System Prompt"),
        "user":      ("#89b4fa", "User"),
        "assistant": ("#a6e3a1", "Assistant"),
    }
    SPEED = 25  # tokens/sec (typical for Llama-3.2-1B on CPU)

    # Count tokens per message
    segments = []
    for m in messages:
        toks = len(model.tokenize(m["content"].encode("utf-8")))
        segments.append({
            "role": m["role"],
            "tokens": toks,
            "preview": m["content"][:60].replace("<", "&lt;").replace(">", "&gt;"),
        })

    total = sum(s["tokens"] for s in segments)
    used_pct = total / n_ctx * 100

    # ── Stacked bar ──────────────────────────────────────────────────
    bar_segs = ""
    for s in segments:
        color = COLORS.get(s["role"], ("#cdd6f4", s["role"]))[0]
        w = max(0.5, s["tokens"] / n_ctx * 100)
        bar_segs += (
            '<div title="' + s["role"] + ': &quot;' + s["preview"] + '&quot; ('
            + str(s["tokens"]) + ' tok)" '
            'style="width:' + str(w) + '%;background:' + color + ';height:100%;'
            'display:inline-block;border-right:1px solid #1e1e2e"></div>'
        )
    remain_pct = max(0, 100 - used_pct)
    bar_segs += (
        '<div style="width:' + str(remain_pct) + '%;background:#313244;height:100%;'
        'display:inline-block;opacity:0.4"></div>'
    )

    # ── Legend ───────────────────────────────────────────────────────
    role_counts = {}
    for s in segments:
        role_counts[s["role"]] = role_counts.get(s["role"], 0) + s["tokens"]
    legend = ""
    for role, toks in role_counts.items():
        color, label = COLORS.get(role, ("#cdd6f4", role))
        legend += (
            '<span style="display:inline-flex;align-items:center;gap:5px;margin-right:14px">'
            '<span style="width:12px;height:12px;background:' + color + ';border-radius:2px;display:inline-block"></span>'
            '<span style="color:#cdd6f4;font-size:0.82em">' + label + ': ' + str(toks) + ' tok</span>'
            '</span>'
        )

    # ── Per-message rows ─────────────────────────────────────────────
    rows = ""
    for s in segments:
        color, label = COLORS.get(s["role"], ("#cdd6f4", s["role"]))
        bar_w = max(2, int(s["tokens"] / total * 280)) if total else 2
        rows += (
            '<tr style="border-bottom:1px solid #313244">'
            '<td style="padding:5px 10px;color:' + color + ';font-weight:bold;width:90px;font-size:0.82em">' + label + '</td>'
            '<td style="padding:5px 10px;color:#a6adc8;font-size:0.78em;max-width:280px;'
            'white-space:nowrap;overflow:hidden;text-overflow:ellipsis">' + s["preview"] + '...</td>'
            '<td style="padding:5px 10px;text-align:right">'
            '<div style="display:inline-block;width:' + str(bar_w) + 'px;height:8px;background:' + color + ';'
            'border-radius:3px;vertical-align:middle;margin-right:6px"></div>'
            '<span style="color:' + color + ';font-size:0.82em">' + str(s["tokens"]) + '</span></td>'
            '</tr>'
        )

    # ── Stat values ──────────────────────────────────────────────────
    pct_str   = "{:.1f}".format(used_pct)
    wait_str  = "{:.1f}".format(total / SPEED)
    remain_tok = n_ctx - total
    fill_color = "#f38ba8" if used_pct > 75 else "#f9e2af" if used_pct > 40 else "#a6e3a1"

    html = (
        '<div style="background:#1e1e2e;border-radius:12px;padding:18px 20px;margin:12px 0">'
        '<h4 style="color:#cdd6f4;margin:0 0 14px 0">' + title + '</h4>'
        # Bar
        '<div style="width:100%;height:22px;border-radius:6px;overflow:hidden;'
        'border:1px solid #45475a;margin-bottom:8px">' + bar_segs + '</div>'
        # Legend
        '<div style="margin-bottom:12px">' + legend
        + '<span style="color:#585b70;font-size:0.82em">| Empty: ' + str(remain_tok) + ' tok remaining</span></div>'
        # Stats row
        '<div style="display:flex;gap:10px;margin-bottom:14px">'
        # Stat 1
        '<div style="flex:1;background:#313244;border-radius:8px;padding:10px 14px;text-align:center">'
        '<div style="color:#a6adc8;font-size:0.75em">Total tokens used</div>'
        '<div style="color:#cdd6f4;font-weight:bold;font-size:1.4em">' + str(total) + '</div></div>'
        # Stat 2
        '<div style="flex:1;background:#313244;border-radius:8px;padding:10px 14px;text-align:center">'
        '<div style="color:#a6adc8;font-size:0.75em">Context filled</div>'
        '<div style="color:' + fill_color + ';font-weight:bold;font-size:1.4em">' + pct_str + '%</div></div>'
        # Stat 3
        '<div style="flex:1;background:#313244;border-radius:8px;padding:10px 14px;text-align:center">'
        '<div style="color:#a6adc8;font-size:0.75em">Est. generation wait</div>'
        '<div style="color:#f9e2af;font-weight:bold;font-size:1.4em">~' + wait_str + 's</div></div>'
        # Stat 4
        '<div style="flex:1;background:#313244;border-radius:8px;padding:10px 14px;text-align:center">'
        '<div style="color:#a6adc8;font-size:0.75em">Messages</div>'
        '<div style="color:#cdd6f4;font-weight:bold;font-size:1.4em">' + str(len(messages)) + '</div></div>'
        '</div>'
        # Table
        '<table style="width:100%;border-collapse:collapse">'
        '<tr style="color:#585b70;font-size:0.78em;border-bottom:1px solid #45475a">'
        '<th style="text-align:left;padding:4px 10px">Role</th>'
        '<th style="text-align:left;padding:4px 10px">Content preview</th>'
        '<th style="text-align:right;padding:4px 10px">Tokens</th></tr>'
        + rows +
        '</table></div>'
    )
    display(HTML(html))


# ── Three snapshots showing context filling up ────────────────────────
print("📊 CONTEXT WINDOW EVOLUTION — Watch it fill up across 3 scenarios\n")

visualize_context_window(
    [{"role": "user", "content": "How do I read a CSV file in Python?"}],
    title="① Bare question (no history, no system prompt)"
)

visualize_context_window(
    [
        {"role": "system", "content": build_course_prompt("Data 100")},
        {"role": "user",   "content": "How do I read a CSV file in Python?"},
    ],
    title="② + System prompt (Data 100 TA persona)"
)

full_msgs = (
    [{"role": "system", "content": "You are a helpful AI assistant for Zoe, a student from Taiwan studying AI at UC Berkeley."}]
    + ZOE_HISTORY
    + [{"role": "user", "content": "How do I read a CSV file in Python?"}]
)
visualize_context_window(full_msgs, title="③ + Zoe's full 25-turn history")


📊 CONTEXT WINDOW EVOLUTION — Watch it fill up across 3 scenarios



Role,Content preview,Tokens
User,How do I read a CSV file in Python?...,11


Role,Content preview,Tokens
System Prompt,You are a helpful AI teaching assistant for Data 100. Your j...,40
User,How do I read a CSV file in Python?...,11


Role,Content preview,Tokens
System Prompt,"You are a helpful AI assistant for Zoe, a student from Taiwa...",20
User,Hi! I'm Zoe. I'm from Taiwan and I'm studying at UC Berkeley...,19
Assistant,Welcome Zoe! Great to meet you. What are you studying?...,14
User,I'm focusing on AI and machine learning. It's my first semes...,17
Assistant,That's exciting! Berkeley has a great CS program. What draws...,18
User,I want to understand how language models work — especially m...,15
Assistant,Context management is a fascinating area. Are you working on...,15
User,"Yes, I'm building a teaching notebook about LLM context mana...",18
Assistant,That sounds like a great project. What tools are you using?...,14
User,"Python, Jupyter, and llama-cpp-python with a local Llama mod...",18



# 🗺️ Part 6: "How Would This Actually Work in Production?"

The assistant works. Prof. Eric is ready to deploy it to all 300 students. But Zoe realises: everything in this notebook runs in a single Python process, in memory, for one user at a time.

She needs to answer Prof. Eric's final question:

> *"What does this look like as a real system — one that can handle hundreds of students simultaneously, persist their profiles across sessions, and stay fast?"*

Every piece of this notebook maps directly to a real backend component. The table and diagram below show how.


In [33]:
from IPython.display import display, HTML

part6_html = """
<div style="background:#1e1e2e;border-radius:12px;padding:20px;font-family:sans-serif">

  <h3 style="color:#cdd6f4;margin:0 0 6px 0">🗺️ Part 6: What Would This Look Like for 300 Students?</h3>
  <p style="color:#a6adc8;font-size:0.9em;margin:0 0 18px 0">
    Zoe's notebook works great — but only for one person at a time. Prof. Eric needs it for 300 students simultaneously.<br>
    Think of it like upgrading from <strong style="color:#cdd6f4">studying alone at home</strong> to running a <strong style="color:#cdd6f4">full classroom</strong>.
  </p>

  <!-- ASCII diagram -->
  <h4 style="color:#f9e2af;margin:0 0 8px 0">📐 System Architecture</h4>
  <pre style="background:#0d0d1a;border:1px solid #45475a;border-radius:8px;
              padding:16px;color:#a6e3a1;font-size:0.82em;line-height:1.8">
  🧑‍💻 Student's Browser
         |
         v
  +--------------------+
  |   Classroom Door   |  ← Checks who you are and if you're allowed in (Auth)
  +--------+-----------+
           |
           v
  +--------------------+
  |   TA / API Server  |  ← Receives the question, decides how to respond
  +--+-----------+-----+
     |           |
     v           v
  📒 Sticky note   📚 Notebook
     (Redis)       (PostgreSQL)
  Last 10 turns    Full learning history
  Fast but temporary  Slower but permanent
     |           |
     v           v
  +--------------------+
  |     AI Model       |  ← Receives full context, generates a reply
  +--------------------+
         |
         v
  📬 Answer streams back to the student
     (one token at a time)
  </pre>

  <!-- Component table - simplified -->
  <h4 style="color:#f9e2af;margin:18px 0 8px 0">📋 Notebook vs Real System</h4>
  <table style="width:100%;border-collapse:collapse;font-size:0.86em">
    <tr style="color:#585b70;border-bottom:1px solid #45475a">
      <th style="text-align:left;padding:8px 10px">In this notebook</th>
      <th style="text-align:left;padding:8px 10px">In a real system</th>
      <th style="text-align:left;padding:8px 10px">School analogy</th>
    </tr>
    <tr style="border-bottom:1px solid #313244">
      <td style="padding:8px 10px;font-family:monospace;color:#89b4fa;font-size:0.85em">recent_history = []</td>
      <td style="padding:8px 10px;color:#a6e3a1">Short-term cache (Redis)</td>
      <td style="padding:8px 10px;color:#a6adc8">Scratch paper during class — thrown away after</td>
    </tr>
    <tr style="border-bottom:1px solid #313244">
      <td style="padding:8px 10px;font-family:monospace;color:#89b4fa;font-size:0.85em">summary = compress(...)</td>
      <td style="padding:8px 10px;color:#a6e3a1">Long-term memory DB (PostgreSQL)</td>
      <td style="padding:8px 10px;color:#a6adc8">Your study notes before finals — kept forever</td>
    </tr>
    <tr style="border-bottom:1px solid #313244">
      <td style="padding:8px 10px;font-family:monospace;color:#89b4fa;font-size:0.85em">retrieve() in RAG</td>
      <td style="padding:8px 10px;color:#a6e3a1">Knowledge search (Vector DB)</td>
      <td style="padding:8px 10px;color:#a6adc8">Going to the library to look something up</td>
    </tr>
    <tr style="border-bottom:1px solid #313244">
      <td style="padding:8px 10px;font-family:monospace;color:#89b4fa;font-size:0.85em">model.create_chat_completion()</td>
      <td style="padding:8px 10px;color:#a6e3a1">AI model server (vLLM)</td>
      <td style="padding:8px 10px;color:#a6adc8">The professor who actually answers the question</td>
    </tr>
    <tr style="border-bottom:1px solid #313244">
      <td style="padding:8px 10px;font-family:monospace;color:#89b4fa;font-size:0.85em">user_profile = {}</td>
      <td style="padding:8px 10px;color:#a6e3a1">Student profile database</td>
      <td style="padding:8px 10px;color:#a6adc8">The school's student records system</td>
    </tr>
    <tr>
      <td style="padding:8px 10px;font-family:monospace;color:#89b4fa;font-size:0.85em">INJECTION_PATTERNS guard</td>
      <td style="padding:8px 10px;color:#a6e3a1">Safety filter (Llama Guard)</td>
      <td style="padding:8px 10px;color:#a6adc8">The security guard at the school entrance</td>
    </tr>
  </table>

  <!-- Key insight -->
  <div style="margin-top:16px;background:#313244;padding:12px 16px;border-radius:8px;
              border-left:4px solid #f9e2af">
    <strong style="color:#f9e2af">💡 Zoe's Takeaway:</strong><br>
    <span style="color:#cdd6f4">
      Every <code style="color:#89b4fa">dict</code> and <code style="color:#89b4fa">list</code> in this notebook
      becomes its own service in a real system.<br>
      <strong>The concepts are identical</strong> — you just swap the simple tools for more powerful ones
      that can handle 300 students at once.
    </span>
  </div>

</div>
"""

display(HTML(part6_html))

In this notebook,In a real system,School analogy
recent_history = [],Short-term cache (Redis),Scratch paper during class — thrown away after
summary = compress(...),Long-term memory DB (PostgreSQL),Your study notes before finals — kept forever
retrieve() in RAG,Knowledge search (Vector DB),Going to the library to look something up
model.create_chat_completion(),AI model server (vLLM),The professor who actually answers the question
user_profile = {},Student profile database,The school's student records system
INJECTION_PATTERNS guard,Safety filter (Llama Guard),The security guard at the school entrance


# 📝 Summary: Zoe's Complete LLM Context Engineering Lab

**Notebook developed by** SzuLun Huang (Zoe) | UC Berkeley Data Science  
**Under the guidance of** Prof. Eric Van Dusen

---

## The journey, part by part

| Part | Prof. Eric's request | Zoe's solution |
|------|---------------------|----------------|
| 1 | "Can it remember our conversation?" | `messages` list — pass full history every call |
| 2 | "Can it adapt to 300 different students?" | Profile dict → auto-generated system prompt |
| 3 | "Why is it getting slower?" | History compression — summarise old turns |
| 3b | — | Anti-patterns: what Zoe almost got wrong |
| 3c | "Can it know about our course materials?" | Simple RAG — inject external knowledge |
| 3d | — | Truncation strategies: sliding window vs compression vs extraction |
| 4 | "A student contradicted their earlier profile" | Conflict detection — auto-update profile |
| 5 | "How much context are we using?" | Context window visualizer |
| 6 | "Can this scale to 300 students?" | Production architecture map |

---

## The mental model

```
Student message
    ↓
[ Safety guard ]              ← blocks injection / PII
    ↓
Retrieve RAG docs             ← Part 3c (course materials)
    ↓
Load student profile          ← Part 2
    ↓
Detect profile changes        ← Part 4
    ↓
Assemble context window       ← Part 5 visualizer
    ↓
LLM inference → stream reply  ← Production: vLLM + SSE
    ↓
Persist turn + update profile ← Production: Redis + PostgreSQL
```

**Context management is the invisible backbone of every LLM product you've ever used.**  
Understanding it at this level puts you ahead of 90% of people building with LLMs. 🎉


# 💬 Part 7: Zoe's Finished Assistant — Try It Yourself

This is everything Zoe built, wrapped into one interface.

You're now a student in Prof. Eric's class. The assistant knows your starting profile — but it will update as you chat. Try:
- Telling it you're actually more experienced than it thinks
- Asking something about the course materials
- Having a long conversation and watching the context fill up

Watch the panel on the right — that's the part students never normally see.

> ⚠️ **Prerequisite:** Run all cells above first (especially Part 4's `ChatAssistant` definition).
